In [4]:
# ============================================================
# STEP 1: SETUP & DATA LOADING
# Apple Inc. (AAPL) — McKinsey-Level Valuation Model
# Data Source: Bloomberg Terminal Export
# ============================================================

# Install required libraries
import subprocess
subprocess.run(["pip", "install", "openpyxl", "xlsxwriter", "--quiet"])

import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import (Font, PatternFill, Alignment, Border, Side,
                              GradientFill)
from openpyxl.utils import get_column_letter
from openpyxl.styles.numbers import FORMAT_PERCENTAGE_00
import warnings
warnings.filterwarnings('ignore')

# ── File path (assumes uploaded to Colab root) ─────────────────
SOURCE_FILE = "Apple.xlsx"
OUTPUT_FILE = "Apple_McKinsey_Valuation.xlsx"

# ── Load all sheets ────────────────────────────────────────────
print("Loading Bloomberg data...")
xl = pd.ExcelFile(SOURCE_FILE)

def load(sheet):
    return pd.read_excel(SOURCE_FILE, sheet_name=sheet, header=None)

df_is   = load('Income - As Reported')
df_bs   = load('Bal Sheet - As Reported')
df_cf   = load('Cash Flow - As Reported')
df_cap  = load('CAPEX & Depreciation')
df_prof = load('Profitability')
df_grow = load('Growth')
df_cred = load('Credit')
df_liq  = load('Liquidity')
df_wc   = load('Working Capital')
df_dup  = load('DuPont Analysis')
df_yld  = load('Yield Analysis')
df_esg  = load('ESG - Overview')
df_soc  = load('Social')
df_gov  = load('Governance')
df_sp   = load('Stock price')

# ── Year labels & column indices ──────────────────────────────
YEARS     = ['FY 2016','FY 2017','FY 2018','FY 2019','FY 2020',
             'FY 2021','FY 2022','FY 2023','FY 2024','FY 2025']
N_HIST    = len(YEARS)          # 10 historical years
COL_START = 2                   # Excel col index where FY2016 starts (0-based)

# ── Helper: extract a numeric row from any sheet ───────────────
def get_row(df, row_idx):
    """Return a list of floats for one data row (cols 2-11). '—' → NaN."""
    raw = df.iloc[row_idx, COL_START : COL_START + N_HIST].tolist()
    out = []
    for v in raw:
        if v == '—' or (isinstance(v, float) and np.isnan(v)):
            out.append(np.nan)
        else:
            try:
                out.append(float(v))
            except:
                out.append(np.nan)
    return out

# ── INCOME STATEMENT rows ──────────────────────────────────────
rev_direct   = get_row(df_is, 7)    # Revenue (only FY16-18 populated)
gross_profit = get_row(df_is, 16)
rd_exp       = get_row(df_is, 14)
sga_exp      = get_row(df_is, 15)
ebit         = get_row(df_is, 88)
int_exp      = get_row(df_is, 46)
pretax_inc   = get_row(df_is, 23)
tax_exp      = get_row(df_is, 22)
net_income   = get_row(df_is, 36)
da_is        = get_row(df_is, 45)   # D&A from IS reference section
eps_diluted  = get_row(df_is, 31)
sh_diluted   = get_row(df_is, 32)
div_per_sh   = get_row(df_is, 28)
eff_tax_rate = get_row(df_is, 67)

# ── Reconstruct REVENUE for all 10 years ──────────────────────
# Method: Revenue = CapEx / (CapEx/Sales ratio)   [both from CAPEX sheet]
capex_vals   = get_row(df_cap, 10)   # Capital Expenditures (negative)
capex_sales  = get_row(df_cap, 11)   # CAPEX/Sales %
da_vals      = get_row(df_cap, 5)    # Depreciation Expense
da_sales_pct = get_row(df_cap, 6)    # D&A / Net Sales %

revenue = []
for i in range(N_HIST):
    if not np.isnan(rev_direct[i]):          # FY16-18: use direct value
        revenue.append(rev_direct[i])
    elif not np.isnan(capex_vals[i]) and not np.isnan(capex_sales[i]) and capex_sales[i] != 0:
        revenue.append(abs(capex_vals[i]) / (capex_sales[i] / 100))
    elif not np.isnan(da_vals[i]) and not np.isnan(da_sales_pct[i]) and da_sales_pct[i] != 0:
        revenue.append(da_vals[i] / (da_sales_pct[i] / 100))
    else:
        revenue.append(np.nan)

# COGS = Revenue - Gross Profit
cogs = [revenue[i] - gross_profit[i] if not np.isnan(revenue[i]) and not np.isnan(gross_profit[i]) else np.nan for i in range(N_HIST)]

# ── BALANCE SHEET rows ────────────────────────────────────────
cash_eq      = get_row(df_bs, 7)
mkt_sec_st   = get_row(df_bs, 8)     # Short-term marketable securities
acc_rec      = get_row(df_bs, 9)
inventories  = get_row(df_bs, 10)
curr_assets  = get_row(df_bs, 15)
ppe_net      = get_row(df_bs, 18)
total_assets = get_row(df_bs, 23)
acc_pay      = get_row(df_bs, 25)
curr_liab    = get_row(df_bs, 31)
lt_debt      = get_row(df_bs, 34)
total_liab   = get_row(df_bs, 37)
tot_equity   = get_row(df_bs, 45)
shares_out   = get_row(df_bs, 41)

# Total debt from Credit sheet
df_cred_full = load('Credit')
total_debt   = get_row(df_cred_full, 7)   # SHORT_AND_LONG_TERM_DEBT

# ── CASH FLOW rows ────────────────────────────────────────────
net_inc_cf   = get_row(df_cf, 7)
da_cf        = get_row(df_cf, 8)
cfo          = get_row(df_cf, 19)
capex_cf     = get_row(df_cf, 22)
cfi          = get_row(df_cf, 31)
div_paid     = get_row(df_cf, 33)
buybacks     = get_row(df_cf, 37)
cff          = get_row(df_cf, 46)

# ── FREE CASH FLOW (from Yield sheet) ─────────────────────────
fcf_equity   = get_row(df_yld, 8)    # CFO + CapEx (CapEx is negative)
mktcap_hist  = get_row(df_yld, 9)    # Historical market caps

# ── PROFITABILITY ratios ───────────────────────────────────────
roe          = get_row(df_prof, 6)
roa          = get_row(df_prof, 7)
roic         = get_row(df_prof, 8)   # row 8 in profitability
gross_margin = get_row(df_prof, 9)   # or calculate from IS
ebit_margin  = get_row(df_prof, 10)
ebitda_marg  = get_row(df_prof, 11)
net_marg     = get_row(df_prof, 12)

# ── GROWTH ratios ─────────────────────────────────────────────
rev_growth   = get_row(df_grow, 6)
ebitda_grw   = get_row(df_grow, 7)
ebit_grw     = get_row(df_grow, 8)
eps_grw      = get_row(df_grow, 9)
ni_grw       = get_row(df_grow, 10)

# ── CREDIT ratios ─────────────────────────────────────────────
debt_ebitda  = get_row(df_cred, 9)
net_dbt_ebi  = get_row(df_cred, 10)
int_cov      = get_row(df_cred, 11)
ebitda_vals  = get_row(df_cred, 12)

# ── LIQUIDITY ratios ──────────────────────────────────────────
cash_ratio   = get_row(df_liq, 5)
curr_ratio   = get_row(df_liq, 6)
quick_ratio  = get_row(df_liq, 7)

# ── WORKING CAPITAL ───────────────────────────────────────────
dso          = get_row(df_wc, 6)    # Days Sales Outstanding
inv_days     = get_row(df_wc, 8)    # Inventory Days
dpo          = get_row(df_wc, 10)   # Days Payable Outstanding
ccc          = get_row(df_wc, 12)   # Cash Conversion Cycle (if available)

# ── DUPONT ────────────────────────────────────────────────────
tax_burden   = get_row(df_dup, 6)
int_burden   = get_row(df_dup, 8)
ebit_margin_dp = get_row(df_dup, 9)
asset_turn   = get_row(df_dup, 10)
leverage     = get_row(df_dup, 11)
roe_dupont   = get_row(df_dup, 12)

# ── ESG SCORES ────────────────────────────────────────────────
esg_total    = get_row(df_esg, 6)
esg_env      = get_row(df_esg, 7)
esg_soc      = get_row(df_esg, 8)
esg_gov      = get_row(df_esg, 9)
employees    = get_row(df_esg, 88)
ghg_scope1   = get_row(df_esg, 23)
ghg_scope2   = get_row(df_esg, 24)
ghg_scope3   = get_row(df_esg, 26)
energy_total = get_row(df_esg, 32)
renew_energy = get_row(df_esg, 33)
water_use    = get_row(df_esg, 54)
women_pct    = get_row(df_esg, 76)
board_size   = get_row(df_esg, 107)
women_board  = get_row(df_esg, 123)

# ── MARKET DATA ───────────────────────────────────────────────
current_price  = float(df_sp.iloc[1, 1])         # $254.23
shares_current = float(sh_diluted[-1]) * 1e6     # latest diluted shares (units)
market_cap_cur = current_price * shares_current  # in USD

print("✅ All data loaded successfully.")
print(f"   Revenue FY2025 : ${revenue[-1]/1000:.1f}B")
print(f"   Net Income FY2025: ${net_income[-1]/1000:.1f}B")
print(f"   FCF FY2025    : ${fcf_equity[-1]/1000:.1f}B")
print(f"   Current Price : ${current_price:.2f}")
print(f"   Mkt Cap (live): ${market_cap_cur/1e12:.2f}T")
print(f"   Shares (dil.) : {sh_diluted[-1]/1000:.2f}B")

Loading Bloomberg data...
✅ All data loaded successfully.
   Revenue FY2025 : $416.2B
   Net Income FY2025: $112.0B
   FCF FY2025    : $98.8B
   Current Price : $254.23
   Mkt Cap (live): $3.81T
   Shares (dil.) : 15.00B


In [5]:
# ============================================================
# STEP 2: DCF ENGINE — WACC, FCFF, TERMINAL VALUE, SENSITIVITY
# ============================================================

# ── WACC INPUTS (McKinsey standard build-up) ─────────────────
# All assumptions clearly labeled — blue cells in final Excel

# Risk-Free Rate: US 10Y Treasury yield (as of Mar 2026)
rf_rate        = 0.0430    # 4.30%

# Equity Risk Premium: Damodaran US ERP
erp            = 0.0480    # 4.80%

# Apple Beta (5Y monthly regression vs S&P 500)
beta           = 1.24

# Cost of Equity via CAPM
ke             = rf_rate + beta * erp

# Pre-tax Cost of Debt: Apple's blended coupon on outstanding bonds
kd_pretax      = 0.0310    # 3.10%

# Effective Tax Rate: FY2025 actual
tax_rate_wacc  = eff_tax_rate[-1] / 100 if not np.isnan(eff_tax_rate[-1]) else 0.156

# After-tax cost of debt
kd_aftertax    = kd_pretax * (1 - tax_rate_wacc)

# Capital Structure weights (based on FY2025 market values)
total_debt_cur = total_debt[-1]           # in $mm
equity_mktcap  = market_cap_cur / 1e6    # convert to $mm
total_capital  = equity_mktcap + total_debt_cur

wd = total_debt_cur  / total_capital      # Debt weight
we = equity_mktcap   / total_capital      # Equity weight

# WACC
wacc = we * ke + wd * kd_aftertax

print("=" * 55)
print("  WACC BUILD-UP")
print("=" * 55)
print(f"  Risk-Free Rate (US 10Y)        : {rf_rate*100:.2f}%")
print(f"  Equity Risk Premium (ERP)      : {erp*100:.2f}%")
print(f"  Beta                           : {beta:.2f}")
print(f"  Cost of Equity (CAPM)          : {ke*100:.2f}%")
print(f"  Pre-Tax Cost of Debt           : {kd_pretax*100:.2f}%")
print(f"  Effective Tax Rate             : {tax_rate_wacc*100:.1f}%")
print(f"  After-Tax Cost of Debt         : {kd_aftertax*100:.2f}%")
print(f"  Debt / Total Capital           : {wd*100:.1f}%")
print(f"  Equity / Total Capital         : {we*100:.1f}%")
print(f"  ─────────────────────────────────────────")
print(f"  WACC                           : {wacc*100:.2f}%")
print()

# ── HISTORICAL FCFF CALCULATION ──────────────────────────────
# FCFF = EBIT × (1 − t) + D&A − ΔWorking Capital − CapEx
# We use: FCFF ≈ CFO + After-tax Interest − CapEx  (Yield sheet basis)

fcff_hist = []
for i in range(N_HIST):
    cfo_v  = cfo[i]
    capex_v = capex_cf[i]        # negative number
    int_v   = int_exp[i]
    t       = (eff_tax_rate[i] / 100) if not np.isnan(eff_tax_rate[i]) else tax_rate_wacc
    if np.isnan(int_v):
        at_int = 0
    else:
        at_int = int_v * (1 - t)
    if not np.isnan(cfo_v) and not np.isnan(capex_v):
        fcff_hist.append(cfo_v + at_int - abs(capex_v))
    else:
        fcff_hist.append(np.nan)

# EBITDA = EBIT + D&A
ebitda_hist = [ebit[i] + da_vals[i] if not np.isnan(ebit[i]) and not np.isnan(da_vals[i]) else np.nan for i in range(N_HIST)]

# NOPAT = EBIT × (1 − tax)
nopat_hist  = [ebit[i] * (1 - (eff_tax_rate[i]/100 if not np.isnan(eff_tax_rate[i]) else tax_rate_wacc))
               for i in range(N_HIST)]

# Invested Capital = Total Assets − Non-interest-bearing current liabilities
# Approx: Total Equity + Total Debt − Cash & ST Securities
inv_cap_hist = []
for i in range(N_HIST):
    ic = tot_equity[i] + total_debt[i] - cash_eq[i]
    if not np.isnan(mkt_sec_st[i]):
        ic -= mkt_sec_st[i]
    inv_cap_hist.append(ic)

# ROIC (recalculated for consistency)
roic_calc = [nopat_hist[i] / inv_cap_hist[i] * 100 if inv_cap_hist[i] != 0 else np.nan for i in range(N_HIST)]

print("  HISTORICAL FCFF ($mm)")
print("  " + "  ".join([f"{y[-4:]}" for y in YEARS]))
print("  " + "  ".join([f"{v/1000:6.1f}B" if not np.isnan(v) else "   N/A " for v in fcff_hist]))
print()

# ── PROJECTION ASSUMPTIONS (5-Year Forecast: FY2026–FY2030) ──
PROJ_YEARS = ['FY 2026E', 'FY 2027E', 'FY 2028E', 'FY 2029E', 'FY 2030E']
N_PROJ = len(PROJ_YEARS)

# Revenue growth rates (McKinsey base case — tapering from Services boom)
rev_growth_proj = [0.085, 0.075, 0.065, 0.055, 0.050]   # 8.5% → 5.0%

# EBITDA margin (expanding on Services mix shift)
ebitda_margin_proj = [0.358, 0.365, 0.370, 0.374, 0.377]  # 35.8% → 37.7%

# D&A as % of revenue (stable, asset-light model)
da_pct_proj    = [0.030, 0.030, 0.029, 0.029, 0.028]

# CapEx as % of revenue (disciplined investment)
capex_pct_proj = [0.030, 0.030, 0.029, 0.028, 0.027]

# Change in NWC as % of revenue (negative = cash inflow, Services model)
nwc_chg_pct    = [-0.005, -0.004, -0.004, -0.003, -0.003]

# Effective tax rate (post OECD global minimum tax)
tax_proj       = [0.160, 0.160, 0.160, 0.160, 0.160]

# Terminal growth rate
tgr = 0.030     # 3.0% — in line with nominal GDP

# ── 5-YEAR PROJECTED INCOME STATEMENT & FCFF ─────────────────
base_rev = revenue[-1]    # FY2025 revenue as anchor

proj_rev    = []
proj_ebitda = []
proj_da     = []
proj_ebit   = []
proj_nopat  = []
proj_capex  = []
proj_nwc    = []
proj_fcff   = []

for i in range(N_PROJ):
    g   = rev_growth_proj[i]
    rev = base_rev * np.prod([1 + rev_growth_proj[j] for j in range(i+1)])
    ebitda = rev * ebitda_margin_proj[i]
    da     = rev * da_pct_proj[i]
    ebit_p = ebitda - da
    nopat  = ebit_p * (1 - tax_proj[i])
    capex  = rev * capex_pct_proj[i]
    nwc    = rev * nwc_chg_pct[i]
    fcff   = nopat + da - capex - nwc

    proj_rev.append(rev)
    proj_ebitda.append(ebitda)
    proj_da.append(da)
    proj_ebit.append(ebit_p)
    proj_nopat.append(nopat)
    proj_capex.append(capex)
    proj_nwc.append(nwc)
    proj_fcff.append(fcff)

# ── DCF VALUATION ─────────────────────────────────────────────
# PV of each projected FCFF
pv_fcff = [proj_fcff[i] / (1 + wacc)**(i+1) for i in range(N_PROJ)]
sum_pv_fcff = sum(pv_fcff)

# Terminal Value (Gordon Growth Model on Year 5 FCFF)
tv = proj_fcff[-1] * (1 + tgr) / (wacc - tgr)
pv_tv = tv / (1 + wacc)**N_PROJ

# Enterprise Value
ev_dcf = sum_pv_fcff + pv_tv

# Bridge to Equity Value
net_debt_cur = total_debt[-1] - cash_eq[-1] - (mkt_sec_st[-1] if not np.isnan(mkt_sec_st[-1]) else 0)
equity_val   = ev_dcf - net_debt_cur

# Per Share (using FY2025 diluted shares in millions)
shares_mm    = sh_diluted[-1]   # in millions (already adjusted)
intrinsic_ps = equity_val / shares_mm

# ── TRADING COMPARABLES (Peer multiples — Big Tech) ──────────
# EV/EBITDA | EV/EBIT | P/E  for valuation cross-check
peer_ev_ebitda = {'MSFT':26.2, 'GOOGL':18.4, 'META':16.8, 'AMZN':20.5, 'AAPL_hist':22.1}
peer_pe        = {'MSFT':33.5, 'GOOGL':22.1, 'META':24.3, 'AMZN':35.2, 'AAPL_hist':30.5}

# AAPL NTM EBITDA (FY2026E)
ntm_ebitda = proj_ebitda[0]
ntm_ebit   = proj_ebit[0]
ntm_eps    = (proj_nopat[0] - (int_exp[-1] if not np.isnan(int_exp[-1]) else 3933) * (1-tax_proj[0])) / shares_mm

comps_ev_ebitda = {k: v * ntm_ebitda for k, v in peer_ev_ebitda.items()}
comps_eq_pe     = {k: v * ntm_eps    for k, v in peer_pe.items()}

# ── SENSITIVITY TABLE: WACC vs Terminal Growth Rate ──────────
wacc_range = [wacc - 0.01, wacc - 0.005, wacc, wacc + 0.005, wacc + 0.01]
tgr_range  = [tgr - 0.01, tgr - 0.005, tgr, tgr + 0.005, tgr + 0.01]

sens_wacc_tgr = {}
for w in wacc_range:
    for t in tgr_range:
        tv_s   = proj_fcff[-1] * (1 + t) / (w - t)
        pv_s   = sum([proj_fcff[i] / (1+w)**(i+1) for i in range(N_PROJ)])
        ev_s   = pv_s + tv_s / (1+w)**N_PROJ
        eq_s   = (ev_s - net_debt_cur) / shares_mm
        sens_wacc_tgr[(round(w,4), round(t,4))] = round(eq_s, 2)

# ── SENSITIVITY TABLE: Revenue Growth vs EBITDA Margin ───────
rev_g_range  = [0.060, 0.070, 0.080, 0.090, 0.100]   # Base case centered
marg_range   = [0.330, 0.345, 0.358, 0.370, 0.385]

sens_rev_marg = {}
for rg in rev_g_range:
    for mg in marg_range:
        r_fcff = []
        b      = base_rev
        for i in range(N_PROJ):
            rv  = b * (1 + rg)**(i+1)
            eb  = rv * mg
            da_ = rv * da_pct_proj[i]
            np_ = (eb - da_) * (1 - tax_proj[i])
            cp_ = rv * capex_pct_proj[i]
            nw_ = rv * nwc_chg_pct[i]
            r_fcff.append(np_ + da_ - cp_ - nw_)
        tv_s   = r_fcff[-1] * (1+tgr) / (wacc - tgr)
        pv_s   = sum([r_fcff[i] / (1+wacc)**(i+1) for i in range(N_PROJ)])
        ev_s   = pv_s + tv_s / (1+wacc)**N_PROJ
        eq_s   = (ev_s - net_debt_cur) / shares_mm
        sens_rev_marg[(round(rg,3), round(mg,3))] = round(eq_s, 2)

# ── SUMMARY PRINTOUT ──────────────────────────────────────────
tv_pct = pv_tv / ev_dcf * 100
print("=" * 55)
print("  DCF VALUATION SUMMARY")
print("=" * 55)
print(f"  WACC                           : {wacc*100:.2f}%")
print(f"  Terminal Growth Rate           : {tgr*100:.1f}%")
print(f"  PV of FCFFs (5-yr)             : ${sum_pv_fcff/1000:,.0f}B")
print(f"  PV of Terminal Value           : ${pv_tv/1000:,.0f}B")
print(f"  TV as % of Enterprise Value    : {tv_pct:.1f}%")
print(f"  Enterprise Value (DCF)         : ${ev_dcf/1000:,.0f}B")
print(f"  (−) Net Debt                   : ${net_debt_cur/1000:,.0f}B")
print(f"  Equity Value                   : ${equity_val/1000:,.0f}B")
print(f"  Diluted Shares                 : {shares_mm/1000:.2f}B")
print(f"  ─────────────────────────────────────────────")
print(f"  Intrinsic Value Per Share (DCF): ${intrinsic_ps:.2f}")
print(f"  Current Market Price           : ${current_price:.2f}")
updown = (intrinsic_ps / current_price - 1) * 100
arrow  = "▲" if updown > 0 else "▼"
print(f"  Upside / (Downside)            : {arrow} {abs(updown):.1f}%")
print()
print("  PROJECTED FINANCIALS ($mm)")
print(f"  {'':12s} " + "  ".join([f"{y}" for y in PROJ_YEARS]))
print(f"  {'Revenue':12s} " + "  ".join([f"${v/1000:7.1f}B" for v in proj_rev]))
print(f"  {'EBITDA':12s} " + "  ".join([f"${v/1000:7.1f}B" for v in proj_ebitda]))
print(f"  {'FCFF':12s} " + "  ".join([f"${v/1000:7.1f}B" for v in proj_fcff]))
print()
print("  WACC × TGR SENSITIVITY (Intrinsic Value per Share)")
header = "WACC\\TGR |" + " | ".join([f"{t*100:.1f}%" for t in tgr_range])
print("  " + header)
for w in wacc_range:
    row = f"{w*100:.2f}%  |"
    for t in tgr_range:
        val = sens_wacc_tgr[(round(w,4), round(t,4))]
        row += f" ${val:>7.2f} |"
    print("  " + row)
print()
print("✅ Step 2 complete — DCF engine built. Ready for Step 3 (Excel writing).")

  WACC BUILD-UP
  Risk-Free Rate (US 10Y)        : 4.30%
  Equity Risk Premium (ERP)      : 4.80%
  Beta                           : 1.24
  Cost of Equity (CAPM)          : 10.25%
  Pre-Tax Cost of Debt           : 3.10%
  Effective Tax Rate             : 15.6%
  After-Tax Cost of Debt         : 2.62%
  Debt / Total Capital           : 2.9%
  Equity / Total Capital         : 97.1%
  ─────────────────────────────────────────
  WACC                           : 10.03%

  HISTORICAL FCFF ($mm)
  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025
    54.2B    52.9B    68.0B    61.9B    75.8B    95.2B   113.9B   102.9B   108.8B    98.8B

  DCF VALUATION SUMMARY
  WACC                           : 10.03%
  Terminal Growth Rate           : 3.0%
  PV of FCFFs (5-yr)             : $557B
  PV of Terminal Value           : $1,545B
  TV as % of Enterprise Value    : 73.5%
  Enterprise Value (DCF)         : $2,102B
  (−) Net Debt                   : $58B
  Equity Value                   : $2,

In [6]:
# ============================================================
# STEP 3: EXCEL ENGINE — BUILD FULL McKINSEY VALUATION BOOK
# ============================================================

from openpyxl import Workbook
from openpyxl.styles import (Font, PatternFill, Alignment, Border, Side)
from openpyxl.utils import get_column_letter
from openpyxl.chart import BarChart, LineChart, Reference
from openpyxl.chart.series import DataPoint
from openpyxl.formatting.rule import ColorScaleRule, DataBarRule
from openpyxl.styles.numbers import (FORMAT_NUMBER_COMMA_SEPARATED1,
                                      FORMAT_PERCENTAGE_00)
from copy import copy
import datetime

wb = Workbook()
wb.remove(wb.active)   # remove default sheet

# ══════════════════════════════════════════════════════════════
# MASTER STYLE SYSTEM
# ══════════════════════════════════════════════════════════════

# ── Palette ───────────────────────────────────────────────────
C_NAVY      = "1B3A6B"   # McKinsey deep navy — section headers
C_DARK_BLUE = "2E5FA3"   # mid-blue — sub-headers
C_LIGHT_BLU = "D6E4F0"   # very light blue — alternating rows
C_ACCENT    = "E8B84B"   # gold accent — highlights / KPIs
C_GREEN     = "217346"   # positive / upside
C_RED       = "C00000"   # negative / downside
C_GREY_HDR  = "F2F2F2"   # light grey — table interior headers
C_WHITE     = "FFFFFF"
C_BLACK     = "000000"
C_BLUE_TXT  = "0000FF"   # hardcoded inputs (industry convention)
C_GREEN_TXT = "008000"   # cross-sheet links
C_BORDER    = "BDD7EE"   # thin border colour

# ── Border helpers ────────────────────────────────────────────
def thin_border(top=True, bottom=True, left=True, right=True):
    s = Side(style='thin', color=C_BORDER)
    n = Side(style=None)
    return Border(
        top    = s if top    else n,
        bottom = s if bottom else n,
        left   = s if left   else n,
        right  = s if right  else n
    )

def thick_border_bottom():
    return Border(bottom=Side(style='medium', color=C_NAVY))

def outer_border():
    s = Side(style='medium', color=C_NAVY)
    return Border(top=s, bottom=s, left=s, right=s)

# ── Fill helpers ──────────────────────────────────────────────
def fill(hex_color):
    return PatternFill("solid", fgColor=hex_color)

# ── Font helpers ──────────────────────────────────────────────
def font(name="Calibri", size=10, bold=False, color=C_BLACK, italic=False):
    return Font(name=name, size=size, bold=bold, color=color, italic=italic)

# ── Alignment shortcuts ───────────────────────────────────────
AL_CTR  = Alignment(horizontal='center',  vertical='center', wrap_text=False)
AL_LEFT = Alignment(horizontal='left',    vertical='center')
AL_RT   = Alignment(horizontal='right',   vertical='center')
AL_WRAP = Alignment(horizontal='left',    vertical='center', wrap_text=True)

# ── Number formats ────────────────────────────────────────────
FMT_MM    = '#,##0;(#,##0);"-"'          # $ millions with parens negatives
FMT_MM_D  = '#,##0.0;(#,##0.0);"-"'     # 1dp millions
FMT_PCT1  = '0.0%;(0.0%);"-"'
FMT_PCT2  = '0.00%;(0.00%);"-"'
FMT_MULT  = '0.0"x";(0.0"x");"-"'
FMT_PRICE = '$#,##0.00'
FMT_PS    = '$#,##0.00'                  # per share
FMT_RATIO = '0.00x;(0.00x);"-"'
FMT_BN    = '#,##0.0"B"'
FMT_INT   = '#,##0;(#,##0);"-"'
FMT_DATE  = 'MMM-YY'

# ══════════════════════════════════════════════════════════════
# CELL WRITER HELPERS
# ══════════════════════════════════════════════════════════════

def w(ws, row, col, value,
      bold=False, size=10, color=C_BLACK, bg=None,
      align=AL_LEFT, fmt=None, italic=False,
      border=None, font_name="Calibri"):
    """Write a single cell with full styling."""
    cell = ws.cell(row=row, column=col, value=value)
    cell.font      = Font(name=font_name, size=size, bold=bold,
                          color=color, italic=italic)
    cell.alignment = align
    if bg:
        cell.fill = fill(bg)
    if fmt:
        cell.number_format = fmt
    if border:
        cell.border = border
    return cell

def section_header(ws, row, col, col_end, text, bg=C_NAVY):
    """Full-width section header bar."""
    ws.merge_cells(start_row=row, start_column=col,
                   end_row=row,   end_column=col_end)
    cell = ws.cell(row=row, column=col, value=text)
    cell.font      = Font(name="Calibri", size=11, bold=True,
                          color=C_WHITE)
    cell.fill      = fill(bg)
    cell.alignment = AL_CTR
    cell.border    = outer_border()
    return cell

def sub_header(ws, row, col, col_end, text, bg=C_DARK_BLUE):
    """Sub-section header."""
    ws.merge_cells(start_row=row, start_column=col,
                   end_row=row,   end_column=col_end)
    cell = ws.cell(row=row, column=col, value=text)
    cell.font      = Font(name="Calibri", size=10, bold=True,
                          color=C_WHITE)
    cell.fill      = fill(bg)
    cell.alignment = AL_CTR
    cell.border    = thin_border()

def col_header(ws, row, col, text, bg=C_GREY_HDR, align=AL_CTR):
    """Column header cell."""
    cell = ws.cell(row=row, column=col, value=text)
    cell.font      = Font(name="Calibri", size=9, bold=True, color=C_NAVY)
    cell.fill      = fill(bg)
    cell.alignment = align
    cell.border    = thin_border()
    return cell

def data_cell(ws, row, col, value, fmt=FMT_MM, is_input=False,
              is_link=False, alt_row=False, is_neg_red=False):
    """Standard data cell following color conventions."""
    cell = ws.cell(row=row, column=col, value=value)
    txt_color = C_BLACK
    if is_input: txt_color = C_BLUE_TXT
    if is_link:  txt_color = C_GREEN_TXT
    bg_color  = C_LIGHT_BLU if alt_row else C_WHITE
    if value is not None and is_neg_red and isinstance(value, (int,float)) and value < 0:
        txt_color = C_RED
    cell.font          = Font(name="Calibri", size=9, color=txt_color)
    cell.fill          = fill(bg_color)
    cell.alignment     = AL_RT
    cell.number_format = fmt
    cell.border        = thin_border()
    return cell

def label_cell(ws, row, col, text, indent=0, alt_row=False, bold=False):
    """Row label cell (left-aligned)."""
    cell = ws.cell(row=row, column=col, value=(" " * indent) + str(text))
    cell.font      = Font(name="Calibri", size=9, bold=bold,
                          color=C_NAVY if bold else C_BLACK)
    cell.fill      = fill(C_LIGHT_BLU if alt_row else C_WHITE)
    cell.alignment = AL_LEFT
    cell.border    = thin_border()
    return cell

def kpi_box(ws, row, col, label, value, fmt, color=C_NAVY, width=2):
    """KPI tile for dashboard."""
    ws.merge_cells(start_row=row,   start_column=col,
                   end_row=row,     end_column=col+width-1)
    ws.merge_cells(start_row=row+1, start_column=col,
                   end_row=row+1,   end_column=col+width-1)
    # Label
    lc = ws.cell(row=row, column=col, value=label)
    lc.font      = Font(name="Calibri", size=9, bold=True, color=C_WHITE)
    lc.fill      = fill(color)
    lc.alignment = AL_CTR
    lc.border    = outer_border()
    # Value
    vc = ws.cell(row=row+1, column=col, value=value)
    vc.font          = Font(name="Calibri", size=14, bold=True,
                            color=color)
    vc.fill          = fill(C_WHITE)
    vc.alignment     = AL_CTR
    vc.number_format = fmt
    vc.border        = outer_border()

def freeze_and_zoom(ws, freeze_cell="B1", zoom=85):
    ws.freeze_panes = freeze_cell
    ws.sheet_view.zoomScale = zoom

def set_col_widths(ws, widths_dict):
    """widths_dict: {col_letter_or_int: width}"""
    for col, w_val in widths_dict.items():
        if isinstance(col, int):
            col = get_column_letter(col)
        ws.column_dimensions[col].width = w_val

def nan_to_none(v):
    """Convert NaN to None for clean Excel output."""
    if isinstance(v, float) and np.isnan(v):
        return None
    return v

def safe(v):
    return nan_to_none(v)

# ══════════════════════════════════════════════════════════════
# SHEET 1 — COVER PAGE
# ══════════════════════════════════════════════════════════════

ws_cov = wb.create_sheet("Cover")
ws_cov.sheet_view.zoomScale = 90
ws_cov.sheet_view.showGridLines = False

# Column widths
for c in range(1, 16):
    ws_cov.column_dimensions[get_column_letter(c)].width = 9.5
ws_cov.column_dimensions['A'].width = 4

# Full navy background band (rows 1-40)
for r in range(1, 6):
    for c in range(1, 16):
        ws_cov.cell(r, c).fill = fill(C_NAVY)

for r in range(6, 28):
    for c in range(1, 16):
        ws_cov.cell(r, c).fill = fill(C_WHITE)

# Gold accent bar
for c in range(1, 16):
    ws_cov.cell(5, c).fill = fill(C_ACCENT)
    ws_cov.row_dimensions[5].height = 6

# Row heights
for r in [1,2,3,4]:
    ws_cov.row_dimensions[r].height = 22
ws_cov.row_dimensions[7].height = 30
ws_cov.row_dimensions[9].height = 50
ws_cov.row_dimensions[11].height = 24

# Title block
ws_cov.merge_cells("B2:O2")
c = ws_cov.cell(2, 2, "APPLE INC. (AAPL US)")
c.font = Font("Calibri", size=22, bold=True, color=C_WHITE)
c.alignment = AL_LEFT

ws_cov.merge_cells("B3:O3")
c = ws_cov.cell(3, 2, "Equity Valuation & Strategic Financial Analysis")
c.font = Font("Calibri", size=13, bold=False, color=C_ACCENT, italic=True)
c.alignment = AL_LEFT

ws_cov.merge_cells("B4:O4")
c = ws_cov.cell(4, 2, f"Prepared by: McKinsey-Style Model  |  Date: {datetime.date.today().strftime('%B %d, %Y')}  |  Source: Bloomberg Terminal")
c.font = Font("Calibri", size=9, color="AAAAAA")
c.alignment = AL_LEFT

# Key metrics in cover
ws_cov.merge_cells("B7:O7")
c = ws_cov.cell(7, 2, "EXECUTIVE SNAPSHOT")
c.font = Font("Calibri", size=14, bold=True, color=C_NAVY)
c.alignment = AL_LEFT

# KPI tiles row 1
kpi_box(ws_cov, 9,  2, "Current Price",       current_price,                       FMT_PRICE, C_NAVY, 2)
kpi_box(ws_cov, 9,  5, "Market Cap",          market_cap_cur/1e9,                  '#,##0.0"B"', C_DARK_BLUE, 2)
kpi_box(ws_cov, 9,  8, "FY2025 Revenue",      revenue[-1]/1000,                    '#,##0.0"B"', C_DARK_BLUE, 2)
kpi_box(ws_cov, 9, 11, "FY2025 Net Income",   net_income[-1]/1000,                 '#,##0.0"B"', C_NAVY, 2)
kpi_box(ws_cov, 9, 14, "FY2025 FCF",          fcf_equity[-1]/1000,                 '#,##0.0"B"', C_DARK_BLUE, 2)

# KPI tiles row 2
kpi_box(ws_cov, 12, 2, "DCF Intrinsic Value", intrinsic_ps,                        FMT_PRICE,
        C_GREEN if intrinsic_ps > current_price else C_RED, 2)
kpi_box(ws_cov, 12, 5, "Upside / (Downside)", (intrinsic_ps/current_price - 1),    FMT_PCT1,
        C_GREEN if intrinsic_ps > current_price else C_RED, 2)
kpi_box(ws_cov, 12, 8, "WACC",                wacc,                                FMT_PCT2, C_NAVY, 2)
kpi_box(ws_cov, 12,11, "FY2025 ROIC",         safe(roic_calc[-1])/100 if safe(roic_calc[-1]) else 0.42,
        FMT_PCT1, C_DARK_BLUE, 2)
kpi_box(ws_cov, 12,14, "ESG Score (BESG)",    safe(esg_total[-3]) or 5.92,         '0.00', C_DARK_BLUE, 2)

# Model navigation table
r = 16
ws_cov.merge_cells(f"B{r}:O{r}")
c = ws_cov.cell(r, 2, "MODEL NAVIGATION")
c.font = Font("Calibri", size=11, bold=True, color=C_NAVY)
c.alignment = AL_LEFT
ws_cov.row_dimensions[r].height = 20

sheets_info = [
    ("1 — Cover",             "This page — executive snapshot & model guide"),
    ("2 — Executive Summary", "KPI dashboard, valuation bridge, FCF waterfall"),
    ("3 — Historical IS",     "Income Statement FY2016–FY2025 (Bloomberg as-reported)"),
    ("4 — Historical BS",     "Balance Sheet FY2016–FY2025 with ratio cross-checks"),
    ("5 — Historical CF",     "Cash Flow Statement FY2016–FY2025 & FCF build"),
    ("6 — DCF Valuation",     "5-year FCFF projection, WACC, TV, equity bridge"),
    ("7 — Sensitivity",       "2D sensitivity: WACC×TGR & Revenue Growth×EBITDA Margin"),
    ("8 — Ratio Dashboard",   "Profitability, Credit, Liquidity, DuPont, Working Capital"),
    ("9 — ESG Scorecard",     "Environmental, Social, Governance scores & trends"),
]

for i, (sname, sdesc) in enumerate(sheets_info):
    row_r = r + 1 + i
    ws_cov.row_dimensions[row_r].height = 16
    alt = (i % 2 == 0)
    bg  = C_LIGHT_BLU if alt else C_WHITE
    c1  = ws_cov.cell(row_r, 2, sname)
    c1.font = Font("Calibri", size=9, bold=True, color=C_NAVY)
    c1.fill = fill(bg); c1.alignment = AL_LEFT; c1.border = thin_border()
    ws_cov.merge_cells(start_row=row_r, start_column=2,
                       end_row=row_r,   end_column=5)
    c2  = ws_cov.cell(row_r, 6, sdesc)
    c2.font = Font("Calibri", size=9, color=C_BLACK)
    c2.fill = fill(bg); c2.alignment = AL_LEFT; c2.border = thin_border()
    ws_cov.merge_cells(start_row=row_r, start_column=6,
                       end_row=row_r,   end_column=15)

# Disclaimer
ws_cov.merge_cells("B27:O27")
c = ws_cov.cell(27, 2,
    "DISCLAIMER: This model is for informational purposes only. "
    "Not investment advice. All data sourced from Bloomberg Terminal. "
    "Projections are estimates based on analyst assumptions.")
c.font      = Font("Calibri", size=8, italic=True, color="888888")
c.alignment = AL_WRAP

print("✅ Sheet 1 (Cover) — done")

# ══════════════════════════════════════════════════════════════
# SHEET 2 — EXECUTIVE SUMMARY / DASHBOARD
# ══════════════════════════════════════════════════════════════

ws_ex = wb.create_sheet("Executive Summary")
ws_ex.sheet_view.showGridLines = False
freeze_and_zoom(ws_ex, "B5", 90)

set_col_widths(ws_ex, {1:28, 2:14, 3:14, 4:14, 5:14,
                        6:14, 7:14, 8:14, 9:14, 10:14, 11:14})

# Title
ws_ex.row_dimensions[1].height = 30
ws_ex.merge_cells("A1:K1")
c = ws_ex.cell(1,1,"APPLE INC. (AAPL US) — EXECUTIVE SUMMARY & VALUATION DASHBOARD")
c.font = Font("Calibri", size=14, bold=True, color=C_WHITE)
c.fill = fill(C_NAVY); c.alignment = AL_CTR

ws_ex.row_dimensions[2].height = 5
for c in range(1,12):
    ws_ex.cell(2,c).fill = fill(C_ACCENT)

# ── KPI Section ───────────────────────────────────────────────
r = 3
section_header(ws_ex, r, 1, 11, "KEY PERFORMANCE INDICATORS — FY2025 ACTUALS & MARKET DATA")
r += 1

kpis = [
    ("Revenue",        revenue[-1],      FMT_MM,   "$mm"),
    ("Gross Profit",   gross_profit[-1], FMT_MM,   "$mm"),
    ("EBITDA",         ebitda_hist[-1],  FMT_MM,   "$mm"),
    ("EBIT",           ebit[-1],         FMT_MM,   "$mm"),
    ("Net Income",     net_income[-1],   FMT_MM,   "$mm"),
    ("Free Cash Flow", fcf_equity[-1],   FMT_MM,   "$mm"),
    ("EPS (Diluted)",  eps_diluted[-1],  FMT_PS,   "$/sh"),
    ("Gross Margin",   gross_profit[-1]/revenue[-1], FMT_PCT1, "%"),
    ("EBIT Margin",    ebit[-1]/revenue[-1],          FMT_PCT1, "%"),
    ("Net Margin",     net_income[-1]/revenue[-1],    FMT_PCT1, "%"),
    ("FCF Yield",      fcf_equity[-1]/mktcap_hist[-1], FMT_PCT1, "%"),
]

for i, (lbl, val, fmt, unit) in enumerate(kpis):
    rr   = r + (i // 4) * 3
    cc   = (i % 4) * 3 + 1
    kpi_box(ws_ex, rr, cc, f"{lbl} ({unit})", safe(val),
            fmt, C_NAVY if i % 2 == 0 else C_DARK_BLUE, 2)

r += 9

# ── Valuation Summary ─────────────────────────────────────────
section_header(ws_ex, r, 1, 11, "VALUATION SUMMARY — MULTI-METHOD TRIANGULATION")
r += 1

sub_header(ws_ex, r, 1, 4, "Method")
sub_header(ws_ex, r, 5, 7, "Implied EV ($mm)")
sub_header(ws_ex, r, 8, 9, "Implied Price/Share")
sub_header(ws_ex, r, 10,11, "Vs. Market Price")
r += 1

val_methods = [
    ("DCF — Base Case (WACC: {:.1f}%, TGR: {:.1f}%)".format(wacc*100, tgr*100),
     ev_dcf, intrinsic_ps),
    ("EV/EBITDA Peer Median ({:.1f}x × NTM EBITDA)".format(np.median(list(peer_ev_ebitda.values()))),
     np.median(list(peer_ev_ebitda.values())) * ntm_ebitda,
     (np.median(list(peer_ev_ebitda.values())) * ntm_ebitda - net_debt_cur) / shares_mm),
    ("P/E Peer Median ({:.1f}x × NTM EPS)".format(np.median(list(peer_pe.values()))),
     (np.median(list(peer_pe.values())) * ntm_eps * shares_mm) + net_debt_cur,
     np.median(list(peer_pe.values())) * ntm_eps),
]

for i, (meth, ev_v, ps_v) in enumerate(val_methods):
    alt = (i % 2 == 0)
    bg  = C_LIGHT_BLU if alt else C_WHITE
    upsid = ps_v / current_price - 1
    # Method label
    c1 = ws_ex.cell(r, 1, meth)
    c1.font = Font("Calibri", 9, color=C_NAVY); c1.fill = fill(bg)
    c1.alignment = AL_LEFT; c1.border = thin_border()
    ws_ex.merge_cells(start_row=r, start_column=1, end_row=r, end_column=4)
    # EV
    c2 = ws_ex.cell(r, 5, safe(ev_v))
    c2.font = Font("Calibri", 9); c2.fill = fill(bg)
    c2.number_format = FMT_MM; c2.alignment = AL_RT; c2.border = thin_border()
    ws_ex.merge_cells(start_row=r, start_column=5, end_row=r, end_column=7)
    # Per share
    c3 = ws_ex.cell(r, 8, safe(ps_v))
    c3.font = Font("Calibri", 9, bold=True); c3.fill = fill(bg)
    c3.number_format = FMT_PS; c3.alignment = AL_CTR; c3.border = thin_border()
    ws_ex.merge_cells(start_row=r, start_column=8, end_row=r, end_column=9)
    # Upside
    color_up = C_GREEN if upsid > 0 else C_RED
    c4 = ws_ex.cell(r, 10, safe(upsid))
    c4.font = Font("Calibri", 9, bold=True, color=color_up)
    c4.fill = fill(bg); c4.number_format = FMT_PCT1
    c4.alignment = AL_CTR; c4.border = thin_border()
    ws_ex.merge_cells(start_row=r, start_column=10, end_row=r, end_column=11)
    r += 1

# Current price reference row
c_pr = ws_ex.cell(r, 1, f"Current Market Price (as of {datetime.date.today().strftime('%d-%b-%Y')})")
c_pr.font = Font("Calibri", 9, bold=True, color=C_NAVY, italic=True)
c_pr.border = thin_border(); c_pr.alignment = AL_LEFT
ws_ex.merge_cells(start_row=r, start_column=1, end_row=r, end_column=7)
cp_v = ws_ex.cell(r, 8, current_price)
cp_v.font = Font("Calibri", 9, bold=True, color=C_NAVY)
cp_v.number_format = FMT_PS; cp_v.alignment = AL_CTR; cp_v.border = thin_border()
ws_ex.merge_cells(start_row=r, start_column=8, end_row=r, end_column=9)
ws_ex.merge_cells(start_row=r, start_column=10, end_row=r, end_column=11)
r += 2

# ── 5-Year Projection Snapshot ────────────────────────────────
section_header(ws_ex, r, 1, 11, "5-YEAR PROJECTION SNAPSHOT ($mm) — BASE CASE")
r += 1

proj_labels = ["", "FY 2025A"] + PROJ_YEARS
for ci, lbl in enumerate(proj_labels):
    col_header(ws_ex, r, ci+1, lbl)
r += 1

proj_rows = [
    ("Revenue ($mm)",       [revenue[-1]]  + proj_rev,    FMT_MM,   False),
    ("  YoY Growth %",      [safe(rev_growth[-1])/100 if safe(rev_growth[-1]) else None]
                           + rev_growth_proj,             FMT_PCT1, False),
    ("EBITDA ($mm)",        [ebitda_hist[-1]] + proj_ebitda, FMT_MM, False),
    ("  EBITDA Margin %",   [ebitda_hist[-1]/revenue[-1]] + ebitda_margin_proj, FMT_PCT1, False),
    ("EBIT ($mm)",          [ebit[-1]]     + proj_ebit,   FMT_MM,   False),
    ("NOPAT ($mm)",         [nopat_hist[-1]] + proj_nopat, FMT_MM,  False),
    ("CapEx ($mm)",         [capex_cf[-1]] + [-v for v in [r*rev for r, rev in zip(capex_pct_proj, proj_rev)]], FMT_MM, False),
    ("Free Cash Flow ($mm)",[fcf_equity[-1]] + proj_fcff, FMT_MM,   False),
    ("  FCF Margin %",      [fcf_equity[-1]/revenue[-1]] + [proj_fcff[i]/proj_rev[i] for i in range(N_PROJ)], FMT_PCT1, False),
]

for ii, (lbl, vals, fmt, inp) in enumerate(proj_rows):
    alt = (ii % 2 == 0)
    bold_row = lbl.startswith("Revenue") or lbl.startswith("Free") or lbl.startswith("EBITDA ($")
    label_cell(ws_ex, r, 1, lbl, indent=0, alt_row=alt, bold=bold_row)
    for ci, v in enumerate(vals):
        data_cell(ws_ex, r, ci+2, safe(v), fmt=fmt, alt_row=alt,
                  is_input=(ci == 0 and not lbl.startswith(" ")))
    r += 1

r += 1
# ── EV Bridge ─────────────────────────────────────────────────
section_header(ws_ex, r, 1, 11, "ENTERPRISE VALUE → EQUITY VALUE BRIDGE ($mm)")
r += 1

bridge = [
    ("PV of FCFFs (FY2026E–FY2030E)",  sum_pv_fcff,    False, False),
    ("PV of Terminal Value",            pv_tv,          False, False),
    ("  TV % of Enterprise Value",      pv_tv/ev_dcf,   True,  False),
    ("Enterprise Value (DCF)",          ev_dcf,         False, True),
    ("(−) Total Debt",                 -total_debt[-1], False, False),
    ("(+) Cash & Equivalents",          cash_eq[-1],    False, False),
    ("Equity Value",                    equity_val,     False, True),
    ("Diluted Shares Outstanding (mm)", shares_mm,      False, False),
    ("Intrinsic Value Per Share",        intrinsic_ps,  False, True),
    ("Current Market Price",            current_price,  False, False),
    ("Upside / (Downside)",             intrinsic_ps/current_price - 1, True, False),
]

col_header(ws_ex, r, 1, "Component", bg=C_GREY_HDR, align=AL_LEFT)
col_header(ws_ex, r, 2, "Value",     bg=C_GREY_HDR)
for c in range(3, 12):
    ws_ex.cell(r, c).fill = fill(C_GREY_HDR)
r += 1

for ii, (lbl, val, is_pct, is_total) in enumerate(bridge):
    alt = ii % 2 == 0
    bold = is_total
    label_cell(ws_ex, r, 1, lbl, indent=0 if is_total else 2, alt_row=alt, bold=bold)
    fmt_  = FMT_PCT1 if is_pct else (FMT_PS if "Share" in lbl or "Price" in lbl else FMT_MM)
    clr   = (C_GREEN if val > 0 else C_RED) if ("Upside" in lbl) else (C_NAVY if is_total else C_BLACK)
    cv    = ws_ex.cell(r, 2, safe(val))
    cv.font = Font("Calibri", 9, bold=bold, color=clr)
    cv.fill = fill(C_LIGHT_BLU if alt else C_WHITE)
    cv.number_format = fmt_; cv.alignment = AL_RT; cv.border = thin_border()
    ws_ex.merge_cells(start_row=r, start_column=2, end_row=r, end_column=11)
    if is_total:
        cv.border = thick_border_bottom()
    r += 1

print("✅ Sheet 2 (Executive Summary) — done")

# ══════════════════════════════════════════════════════════════
# SHEET 3 — HISTORICAL INCOME STATEMENT
# ══════════════════════════════════════════════════════════════

ws_is = wb.create_sheet("Historical IS")
ws_is.sheet_view.showGridLines = False
freeze_and_zoom(ws_is, "C6", 85)

set_col_widths(ws_is, {1:34, **{i:13 for i in range(2,13)}})

# Header
ws_is.row_dimensions[1].height = 28
ws_is.merge_cells("A1:L1")
c = ws_is.cell(1,1,"APPLE INC. — HISTORICAL INCOME STATEMENT ($mm) | FY2016–FY2025 | Source: Bloomberg Terminal")
c.font = Font("Calibri",12,True,color=C_WHITE); c.fill=fill(C_NAVY); c.alignment=AL_CTR

# Gold stripe
ws_is.row_dimensions[2].height = 5
for col in range(1,13): ws_is.cell(2,col).fill = fill(C_ACCENT)

# Column headers
r = 3
col_header(ws_is, r, 1, "Income Statement ($mm)", bg=C_GREY_HDR, align=AL_LEFT)
for i, yr in enumerate(YEARS):
    col_header(ws_is, r, i+2, yr)
r += 1

is_data = [
    # (label, values, fmt, indent, bold, is_total)
    ("REVENUES",                    revenue,                FMT_MM,   0, True,  True),
    ("  Cost of Goods Sold",        cogs,                   FMT_MM,   2, False, False),
    ("GROSS PROFIT",                gross_profit,           FMT_MM,   0, True,  True),
    ("  Gross Margin %",            [gross_profit[i]/revenue[i] if safe(revenue[i]) and revenue[i]!=0 else None for i in range(N_HIST)], FMT_PCT1, 2, False, False),
    ("  R&D Expenditures",          rd_exp,                 FMT_MM,   2, False, False),
    ("  SG&A Expenses",             sga_exp,                FMT_MM,   2, False, False),
    ("EBIT (Operating Income)",     ebit,                   FMT_MM,   0, True,  True),
    ("  EBIT Margin %",             [ebit[i]/revenue[i] if safe(revenue[i]) and revenue[i]!=0 else None for i in range(N_HIST)], FMT_PCT1, 2, False, False),
    ("  Depreciation & Amortization", da_vals,              FMT_MM,   2, False, False),
    ("EBITDA",                      ebitda_hist,            FMT_MM,   0, True,  True),
    ("  EBITDA Margin %",           [ebitda_hist[i]/revenue[i] if safe(revenue[i]) and revenue[i]!=0 else None for i in range(N_HIST)], FMT_PCT1, 2, False, False),
    ("  Interest Expense",          int_exp,                FMT_MM,   2, False, False),
    ("  Other Non-Operating Inc/(Exp)", [None]*N_HIST,      FMT_MM,   2, False, False),
    ("PRE-TAX INCOME",              pretax_inc,             FMT_MM,   0, True,  True),
    ("  Income Tax Expense",        tax_exp,                FMT_MM,   2, False, False),
    ("  Effective Tax Rate %",      [v/100 if safe(v) else None for v in eff_tax_rate], FMT_PCT1, 2, False, False),
    ("NET INCOME",                  net_income,             FMT_MM,   0, True,  True),
    ("  Net Margin %",              [net_income[i]/revenue[i] if safe(revenue[i]) and revenue[i]!=0 else None for i in range(N_HIST)], FMT_PCT1, 2, False, False),
    ("  EPS — Diluted ($/sh)",      eps_diluted,            FMT_PS,   2, False, False),
    ("  Diluted Shares Outstdg (mm)", sh_diluted,           '#,##0.0', 2, False, False),
    ("  Dividends Per Share ($/sh)", div_per_sh,            FMT_PS,   2, False, False),
    ("── REFERENCE ITEMS ──",       [None]*N_HIST,          FMT_MM,   0, False, False),
    ("  Stock-Based Compensation",  get_row(df_is, 60),     FMT_MM,   2, False, False),
    ("  Capital Expenditures",      [abs(v) if safe(v) else None for v in capex_cf], FMT_MM, 2, False, False),
    ("  Free Cash Flow to Equity",  fcf_equity,             FMT_MM,   2, True,  False),
]

section_rows = {"REVENUES", "GROSS PROFIT", "EBIT (Operating Income)",
                "EBITDA", "PRE-TAX INCOME", "NET INCOME"}

for ii, (lbl, vals, fmt, indent, bold, is_tot) in enumerate(is_data):
    alt = (ii % 2 == 0)
    ws_is.row_dimensions[r].height = 15

    if lbl in section_rows:
        ws_is.merge_cells(start_row=r, start_column=1, end_row=r, end_column=1)
        lc = ws_is.cell(r, 1, lbl)
        lc.font = Font("Calibri",9,True,color=C_WHITE)
        lc.fill = fill(C_DARK_BLUE); lc.alignment=AL_LEFT; lc.border=thin_border()
        for ci, v in enumerate(vals):
            dc = ws_is.cell(r, ci+2, safe(v))
            dc.font = Font("Calibri",9,True,color=C_WHITE)
            dc.fill = fill(C_DARK_BLUE)
            dc.number_format = fmt; dc.alignment=AL_RT; dc.border=thin_border()
    elif lbl.startswith("──"):
        for c in range(1,13):
            cc = ws_is.cell(r,c, lbl if c==1 else None)
            cc.fill = fill(C_GREY_HDR)
            cc.font = Font("Calibri",8,True,color=C_NAVY,italic=True)
            cc.border = thin_border()
    else:
        label_cell(ws_is, r, 1, lbl, indent=indent, alt_row=alt, bold=bold)
        for ci, v in enumerate(vals):
            is_margin = "%" in lbl or "Margin" in lbl or "Rate" in lbl
            data_cell(ws_is, r, ci+2, safe(v), fmt=fmt, alt_row=alt,
                      is_neg_red=("Expense" in lbl or "Cost" in lbl))
    r += 1

# Revenue CAGR row
r += 1
section_header(ws_is, r, 1, 12, "REVENUE CAGR ANALYSIS", bg=C_GREY_HDR)
ws_is.cell(r,1).font = Font("Calibri",9,True,color=C_NAVY)
r += 1
ws_is.cell(r,1,"5-Year Revenue CAGR (FY2020–FY2025)").font = Font("Calibri",9)
ws_is.cell(r,1).border = thin_border()
rev_20 = revenue[4]; rev_25 = revenue[-1]
cagr5 = (rev_25/rev_20)**(1/5) - 1 if rev_20 and rev_20!=0 else None
cv = ws_is.cell(r, 2, safe(cagr5))
cv.number_format=FMT_PCT1; cv.font=Font("Calibri",9,bold=True,color=C_GREEN if cagr5 and cagr5>0 else C_RED)
cv.border=thin_border(); cv.alignment=AL_RT

print("✅ Sheet 3 (Historical IS) — done")

# ══════════════════════════════════════════════════════════════
# SHEET 4 — HISTORICAL BALANCE SHEET
# ══════════════════════════════════════════════════════════════

ws_bs = wb.create_sheet("Historical BS")
ws_bs.sheet_view.showGridLines = False
freeze_and_zoom(ws_bs, "C6", 85)
set_col_widths(ws_bs, {1:34, **{i:13 for i in range(2,13)}})

ws_bs.row_dimensions[1].height = 28
ws_bs.merge_cells("A1:L1")
c = ws_bs.cell(1,1,"APPLE INC. — HISTORICAL BALANCE SHEET ($mm) | FY2016–FY2025 | Source: Bloomberg Terminal")
c.font = Font("Calibri",12,True,color=C_WHITE); c.fill=fill(C_NAVY); c.alignment=AL_CTR
ws_bs.row_dimensions[2].height = 5
for col in range(1,13): ws_bs.cell(2,col).fill=fill(C_ACCENT)

r = 3
col_header(ws_bs, r, 1, "Balance Sheet ($mm)", bg=C_GREY_HDR, align=AL_LEFT)
for i, yr in enumerate(YEARS):
    col_header(ws_bs, r, i+2, yr)
r += 1

non_curr_mkt = get_row(df_bs, 17)
goodwill     = get_row(df_bs, 20)
other_nc_ast = get_row(df_bs, 21)
other_curr   = get_row(df_bs, 12)
st_borrow    = get_row(df_bs, 27)
curr_ltd     = get_row(df_bs, 26)
def_rev_st   = get_row(df_bs, 29)
other_cl     = get_row(df_bs, 30)
def_rev_lt   = get_row(df_bs, 35)
other_ncl    = get_row(df_bs, 36)
ret_earn     = get_row(df_bs, 40)
cs_apic      = get_row(df_bs, 43)
aoci         = get_row(df_bs, 39)

bs_data = [
    ("ASSETS",                           [None]*N_HIST,  FMT_MM, 0, True,  True),
    ("── CURRENT ASSETS ──",             [None]*N_HIST,  FMT_MM, 0, False, False),
    ("  Cash & Equivalents",             cash_eq,        FMT_MM, 2, False, False),
    ("  Short-Term Marketable Securities", mkt_sec_st,   FMT_MM, 2, False, False),
    ("  Accounts Receivable",            acc_rec,        FMT_MM, 2, False, False),
    ("  Inventories",                    inventories,    FMT_MM, 2, False, False),
    ("  Other Current Assets",           other_curr,     FMT_MM, 2, False, False),
    ("TOTAL CURRENT ASSETS",             curr_assets,    FMT_MM, 0, True,  True),
    ("── NON-CURRENT ASSETS ──",         [None]*N_HIST,  FMT_MM, 0, False, False),
    ("  Non-Current Marketable Securities", non_curr_mkt, FMT_MM, 2, False, False),
    ("  Property, Plant & Equipment Net",ppe_net,        FMT_MM, 2, False, False),
    ("  Goodwill",                       goodwill,       FMT_MM, 2, False, False),
    ("  Other Non-Current Assets",       other_nc_ast,   FMT_MM, 2, False, False),
    ("TOTAL ASSETS",                     total_assets,   FMT_MM, 0, True,  True),
    ("LIABILITIES & EQUITY",             [None]*N_HIST,  FMT_MM, 0, True,  True),
    ("── CURRENT LIABILITIES ──",        [None]*N_HIST,  FMT_MM, 0, False, False),
    ("  Accounts Payable",               acc_pay,        FMT_MM, 2, False, False),
    ("  Current Portion LT Debt",        curr_ltd,       FMT_MM, 2, False, False),
    ("  Short-Term Borrowings",          st_borrow,      FMT_MM, 2, False, False),
    ("  Deferred Revenue (ST)",          def_rev_st,     FMT_MM, 2, False, False),
    ("  Other Current Liabilities",      other_cl,       FMT_MM, 2, False, False),
    ("TOTAL CURRENT LIABILITIES",        curr_liab,      FMT_MM, 0, True,  True),
    ("── NON-CURRENT LIABILITIES ──",    [None]*N_HIST,  FMT_MM, 0, False, False),
    ("  Long-Term Debt",                 lt_debt,        FMT_MM, 2, False, False),
    ("  Deferred Revenue (LT)",          def_rev_lt,     FMT_MM, 2, False, False),
    ("  Other Non-Current Liabilities",  other_ncl,      FMT_MM, 2, False, False),
    ("TOTAL LIABILITIES",                total_liab,     FMT_MM, 0, True,  True),
    ("── STOCKHOLDERS' EQUITY ──",       [None]*N_HIST,  FMT_MM, 0, False, False),
    ("  Common Stock & APIC",            cs_apic,        FMT_MM, 2, False, False),
    ("  Retained Earnings",              ret_earn,       FMT_MM, 2, False, False),
    ("  Accumulated OCI",                aoci,           FMT_MM, 2, False, False),
    ("TOTAL SHAREHOLDERS' EQUITY",       tot_equity,     FMT_MM, 0, True,  True),
    ("TOTAL LIABILITIES + EQUITY",       total_assets,   FMT_MM, 0, True,  True),
    ("── KEY METRICS ──",                [None]*N_HIST,  FMT_MM, 0, False, False),
    ("  Total Debt (ST+LT)",             total_debt,     FMT_MM, 2, False, False),
    ("  Net Debt (Debt−Cash−ST Sec.)",   inv_cap_hist,   FMT_MM, 2, False, False),
    ("  Debt / Equity Ratio",            [total_debt[i]/tot_equity[i] if safe(tot_equity[i]) and tot_equity[i]!=0 else None for i in range(N_HIST)], FMT_MULT, 2, False, False),
    ("  Book Value Per Share ($)",       [tot_equity[i]/sh_diluted[i] if safe(sh_diluted[i]) and sh_diluted[i]!=0 else None for i in range(N_HIST)], FMT_PS, 2, False, False),
]

hdr_lbls = {"ASSETS","LIABILITIES & EQUITY","TOTAL CURRENT ASSETS",
            "TOTAL ASSETS","TOTAL CURRENT LIABILITIES","TOTAL LIABILITIES",
            "TOTAL SHAREHOLDERS' EQUITY","TOTAL LIABILITIES + EQUITY"}

for ii, (lbl, vals, fmt, indent, bold, is_tot) in enumerate(bs_data):
    alt = ii % 2 == 0
    ws_bs.row_dimensions[r].height = 15
    if lbl in hdr_lbls:
        lc = ws_bs.cell(r,1,lbl)
        lc.font=Font("Calibri",9,True,color=C_WHITE)
        lc.fill=fill(C_DARK_BLUE); lc.alignment=AL_LEFT; lc.border=thin_border()
        for ci, v in enumerate(vals):
            dc = ws_bs.cell(r,ci+2,safe(v))
            dc.font=Font("Calibri",9,True,color=C_WHITE)
            dc.fill=fill(C_DARK_BLUE)
            dc.number_format=fmt; dc.alignment=AL_RT; dc.border=thin_border()
    elif lbl.startswith("──"):
        for c in range(1,13):
            cc=ws_bs.cell(r,c,lbl if c==1 else None)
            cc.fill=fill(C_GREY_HDR)
            cc.font=Font("Calibri",8,True,color=C_NAVY,italic=True)
            cc.border=thin_border()
    else:
        label_cell(ws_bs, r, 1, lbl, indent=indent, alt_row=alt, bold=bold)
        for ci, v in enumerate(vals):
            data_cell(ws_bs, r, ci+2, safe(v), fmt=fmt, alt_row=alt)
    r += 1

print("✅ Sheet 4 (Historical BS) — done")

# ══════════════════════════════════════════════════════════════
# SHEET 5 — HISTORICAL CASH FLOW
# ══════════════════════════════════════════════════════════════

ws_cf = wb.create_sheet("Historical CF")
ws_cf.sheet_view.showGridLines = False
freeze_and_zoom(ws_cf, "C6", 85)
set_col_widths(ws_cf, {1:34, **{i:13 for i in range(2,13)}})

ws_cf.row_dimensions[1].height = 28
ws_cf.merge_cells("A1:L1")
c = ws_cf.cell(1,1,"APPLE INC. — HISTORICAL CASH FLOW STATEMENT ($mm) | FY2016–FY2025 | Source: Bloomberg Terminal")
c.font=Font("Calibri",12,True,color=C_WHITE); c.fill=fill(C_NAVY); c.alignment=AL_CTR
ws_cf.row_dimensions[2].height = 5
for col in range(1,13): ws_cf.cell(2,col).fill=fill(C_ACCENT)

r = 3
col_header(ws_cf, r, 1, "Cash Flow Statement ($mm)", bg=C_GREY_HDR, align=AL_LEFT)
for i, yr in enumerate(YEARS):
    col_header(ws_cf, r, i+2, yr)
r += 1

# Pull additional CF rows
chg_inv      = get_row(df_cf, 12)
chg_ap       = get_row(df_cf, 13)
chg_ar       = get_row(df_cf, 14)
oth_noncash  = get_row(df_cf, 11)
proc_invest  = get_row(df_cf, 24)
pur_invest   = get_row(df_cf, 25)
acq_biz      = get_row(df_cf, 26)
net_chg_cash = get_row(df_cf, 42)
cash_end     = get_row(df_cf, 43)
tax_paid     = get_row(df_cf, 39)
int_paid     = get_row(df_cf, 40)
inc_lt_debt  = get_row(df_cf, 34)
dec_lt_debt  = get_row(df_cf, 35)
iss_stk      = get_row(df_cf, 36)

cf_data = [
    ("OPERATING ACTIVITIES",                 [None]*N_HIST, FMT_MM, 0, True, True),
    ("  Net Income",                         net_inc_cf,    FMT_MM, 2, False, False),
    ("  Depreciation & Amortization",        da_cf,         FMT_MM, 2, False, False),
    ("  Stock-Based Compensation",           get_row(df_cf,10), FMT_MM, 2, False, False),
    ("  Deferred Income Taxes",              get_row(df_cf,9),  FMT_MM, 2, False, False),
    ("  Change in Accounts Receivable",      chg_ar,        FMT_MM, 2, False, False),
    ("  Change in Inventories",              chg_inv,       FMT_MM, 2, False, False),
    ("  Change in Accounts Payable",         chg_ap,        FMT_MM, 2, False, False),
    ("  Other Operating Changes",            oth_noncash,   FMT_MM, 2, False, False),
    ("TOTAL CASH FROM OPERATIONS (CFO)",     cfo,           FMT_MM, 0, True, True),
    ("  CFO Margin %",                       [cfo[i]/revenue[i] if safe(revenue[i]) and revenue[i]!=0 else None for i in range(N_HIST)], FMT_PCT1, 2, False, False),
    ("INVESTING ACTIVITIES",                 [None]*N_HIST, FMT_MM, 0, True, True),
    ("  Capital Expenditures",               capex_cf,      FMT_MM, 2, False, False),
    ("  Proceeds from Investments",          proc_invest,   FMT_MM, 2, False, False),
    ("  Purchases of Investments",           pur_invest,    FMT_MM, 2, False, False),
    ("  Acquisitions",                       acq_biz,       FMT_MM, 2, False, False),
    ("TOTAL CASH FROM INVESTING (CFI)",      cfi,           FMT_MM, 0, True, True),
    ("FINANCING ACTIVITIES",                 [None]*N_HIST, FMT_MM, 0, True, True),
    ("  Dividends Paid",                     div_paid,      FMT_MM, 2, False, False),
    ("  Share Repurchases",                  buybacks,      FMT_MM, 2, False, False),
    ("  Increase in LT Debt",                inc_lt_debt,   FMT_MM, 2, False, False),
    ("  Decrease in LT Debt",                dec_lt_debt,   FMT_MM, 2, False, False),
    ("  Issuance of Common Stock",           iss_stk,       FMT_MM, 2, False, False),
    ("TOTAL CASH FROM FINANCING (CFF)",      cff,           FMT_MM, 0, True, True),
    ("NET CHANGE IN CASH",                   net_chg_cash,  FMT_MM, 0, True, True),
    ("CASH END OF PERIOD",                   cash_end,      FMT_MM, 0, True, True),
    ("── FCF BUILD ──",                      [None]*N_HIST, FMT_MM, 0, False, False),
    ("  CFO",                                cfo,           FMT_MM, 2, False, False),
    ("  Less: Capital Expenditures",         capex_cf,      FMT_MM, 2, False, False),
    ("FREE CASH FLOW TO EQUITY",             fcf_equity,    FMT_MM, 0, True, True),
    ("  FCF Yield (vs. Hist. Mkt Cap)",      [fcf_equity[i]/mktcap_hist[i] if safe(mktcap_hist[i]) and mktcap_hist[i]!=0 else None for i in range(N_HIST)], FMT_PCT1, 2, False, False),
    ("  Cash Paid for Taxes",                tax_paid,      FMT_MM, 2, False, False),
    ("  Cash Paid for Interest",             int_paid,      FMT_MM, 2, False, False),
    ("  Dividends + Buybacks (Total Return)",
     [abs(div_paid[i]) + abs(buybacks[i]) if safe(div_paid[i]) and safe(buybacks[i]) else None for i in range(N_HIST)],
     FMT_MM, 2, True, False),
]

cf_hdr = {"OPERATING ACTIVITIES","INVESTING ACTIVITIES","FINANCING ACTIVITIES",
          "TOTAL CASH FROM OPERATIONS (CFO)","TOTAL CASH FROM INVESTING (CFI)",
          "TOTAL CASH FROM FINANCING (CFF)","NET CHANGE IN CASH","CASH END OF PERIOD",
          "FREE CASH FLOW TO EQUITY"}

for ii, (lbl, vals, fmt, indent, bold, is_tot) in enumerate(cf_data):
    alt = ii % 2 == 0
    ws_cf.row_dimensions[r].height = 15
    if lbl in cf_hdr:
        lc = ws_cf.cell(r,1,lbl)
        lc.font=Font("Calibri",9,True,color=C_WHITE)
        lc.fill=fill(C_DARK_BLUE); lc.alignment=AL_LEFT; lc.border=thin_border()
        for ci,v in enumerate(vals):
            dc=ws_cf.cell(r,ci+2,safe(v))
            dc.font=Font("Calibri",9,True,color=C_WHITE)
            dc.fill=fill(C_DARK_BLUE)
            dc.number_format=fmt; dc.alignment=AL_RT; dc.border=thin_border()
    elif lbl.startswith("──"):
        for c in range(1,13):
            cc=ws_cf.cell(r,c,lbl if c==1 else None)
            cc.fill=fill(C_GREY_HDR)
            cc.font=Font("Calibri",8,True,color=C_NAVY,italic=True)
            cc.border=thin_border()
    else:
        label_cell(ws_cf, r, 1, lbl, indent=indent, alt_row=alt, bold=bold)
        for ci,v in enumerate(vals):
            data_cell(ws_cf, r, ci+2, safe(v), fmt=fmt, alt_row=alt)
    r += 1

print("✅ Sheet 5 (Historical CF) — done")
print()
print("✅ Step 3 complete — 5 sheets built (Cover, Executive Summary, IS, BS, CF).")
print("   Reply 'Step 4' to get the DCF sheet, Sensitivity, and Ratio Dashboard.")

✅ Sheet 1 (Cover) — done
✅ Sheet 2 (Executive Summary) — done
✅ Sheet 3 (Historical IS) — done
✅ Sheet 4 (Historical BS) — done
✅ Sheet 5 (Historical CF) — done

✅ Step 3 complete — 5 sheets built (Cover, Executive Summary, IS, BS, CF).
   Reply 'Step 4' to get the DCF sheet, Sensitivity, and Ratio Dashboard.


In [7]:
# ============================================================
# STEP 4: DCF VALUATION, SENSITIVITY, RATIO DASHBOARD,
#         ESG SCORECARD — FINAL SAVE
# ============================================================

# ══════════════════════════════════════════════════════════════
# SHEET 6 — DCF VALUATION MODEL
# ══════════════════════════════════════════════════════════════

ws_dcf = wb.create_sheet("DCF Valuation")
ws_dcf.sheet_view.showGridLines = False
freeze_and_zoom(ws_dcf, "C7", 85)
set_col_widths(ws_dcf, {1:34, 2:15, 3:15, 4:15, 5:15,
                         6:15, 7:15, 8:15, 9:15, 10:15})

# ── Title ─────────────────────────────────────────────────────
ws_dcf.row_dimensions[1].height = 28
ws_dcf.merge_cells("A1:J1")
c = ws_dcf.cell(1,1,"APPLE INC. — DISCOUNTED CASH FLOW VALUATION MODEL | McKinsey FCFF Approach | $mm")
c.font=Font("Calibri",12,True,color=C_WHITE); c.fill=fill(C_NAVY); c.alignment=AL_CTR
ws_dcf.row_dimensions[2].height = 5
for col in range(1,11): ws_dcf.cell(2,col).fill=fill(C_ACCENT)

# ── WACC BUILD-UP SECTION ─────────────────────────────────────
r = 3
section_header(ws_dcf, r, 1, 10,
    "SECTION 1 — WACC BUILD-UP  |  Blue = Input Assumption  |  Black = Formula")
r += 1

sub_header(ws_dcf, r, 1, 5, "Cost of Equity (CAPM)", bg=C_DARK_BLUE)
sub_header(ws_dcf, r, 6, 10, "Capital Structure & WACC", bg=C_DARK_BLUE)
r += 1

wacc_left = [
    ("Risk-Free Rate (US 10Y Treasury)",   rf_rate,       FMT_PCT2, True),
    ("Equity Risk Premium (Damodaran)",    erp,           FMT_PCT2, True),
    ("Beta (5Y Monthly vs S&P 500)",       beta,          '0.00',   True),
    ("Cost of Equity (CAPM = Rf+β×ERP)",  ke,            FMT_PCT2, False),
    ("Pre-Tax Cost of Debt",               kd_pretax,     FMT_PCT2, True),
    ("Effective Tax Rate",                 tax_rate_wacc, FMT_PCT2, False),
    ("After-Tax Cost of Debt",             kd_aftertax,   FMT_PCT2, False),
]
wacc_right = [
    ("Market Cap ($mm)",                   equity_mktcap, FMT_MM,   False),
    ("Total Debt ($mm)",                   total_debt_cur,FMT_MM,   False),
    ("Total Capital ($mm)",                total_capital, FMT_MM,   False),
    ("Equity Weight (We)",                 we,            FMT_PCT2, False),
    ("Debt Weight (Wd)",                   wd,            FMT_PCT2, False),
    ("WACC  =  We×Ke + Wd×Kd(1-t)",       wacc,          FMT_PCT2, False),
    ("Terminal Growth Rate (TGR)",         tgr,           FMT_PCT2, True),
]

for ii in range(max(len(wacc_left), len(wacc_right))):
    alt = ii % 2 == 0
    bg  = C_LIGHT_BLU if alt else C_WHITE
    if ii < len(wacc_left):
        lbl, val, fmt, is_inp = wacc_left[ii]
        is_wacc_row = "WACC" in lbl
        lc = ws_dcf.cell(r, 1, lbl)
        lc.font  = Font("Calibri", 9, bold=is_wacc_row,
                         color=C_NAVY if is_wacc_row else C_BLACK)
        lc.fill  = fill(C_LIGHT_BLU if is_wacc_row else bg)
        lc.alignment = AL_LEFT; lc.border = thin_border()
        vc = ws_dcf.cell(r, 2, val)
        vc.font  = Font("Calibri", 9, bold=is_wacc_row,
                         color=C_BLUE_TXT if is_inp else
                              (C_NAVY if is_wacc_row else C_BLACK))
        vc.fill  = fill(C_LIGHT_BLU if is_wacc_row else bg)
        vc.number_format = fmt; vc.alignment = AL_RT; vc.border = thin_border()
        ws_dcf.merge_cells(start_row=r, start_column=1,
                            end_row=r,   end_column=4)
        ws_dcf.merge_cells(start_row=r, start_column=5,
                            end_row=r,   end_column=5)
    if ii < len(wacc_right):
        lbl2, val2, fmt2, is_inp2 = wacc_right[ii]
        is_wacc2 = "WACC" in lbl2 and "=" in lbl2
        lc2 = ws_dcf.cell(r, 6, lbl2)
        lc2.font  = Font("Calibri", 9, bold=is_wacc2,
                          color=C_NAVY if is_wacc2 else C_BLACK)
        lc2.fill  = fill(C_LIGHT_BLU if is_wacc2 else bg)
        lc2.alignment = AL_LEFT; lc2.border = thin_border()
        vc2 = ws_dcf.cell(r, 7, val2)
        vc2.font  = Font("Calibri", 9, bold=is_wacc2,
                          color=C_BLUE_TXT if is_inp2 else
                               (C_NAVY if is_wacc2 else C_BLACK))
        vc2.fill  = fill(C_LIGHT_BLU if is_wacc2 else bg)
        vc2.number_format = fmt2; vc2.alignment = AL_RT; vc2.border = thin_border()
        ws_dcf.merge_cells(start_row=r, start_column=6,
                            end_row=r,   end_column=9)
        ws_dcf.merge_cells(start_row=r, start_column=9,
                            end_row=r,   end_column=10)
    r += 1

r += 1

# ── PROJECTION ASSUMPTIONS ────────────────────────────────────
section_header(ws_dcf, r, 1, 10,
    "SECTION 2 — PROJECTION ASSUMPTIONS (FY2026E–FY2030E)  |  Blue = Input")
r += 1

all_years_dcf = ["FY 2025A"] + PROJ_YEARS
col_header(ws_dcf, r, 1, "Assumption Driver", bg=C_GREY_HDR, align=AL_LEFT)
for i, yr in enumerate(all_years_dcf):
    col_header(ws_dcf, r, i+2, yr)
r += 1

assump_rows = [
    ("Revenue Growth Rate %",
     [safe(rev_growth[-1])/100 if safe(rev_growth[-1]) else None]
     + rev_growth_proj, FMT_PCT2, True),
    ("EBITDA Margin %",
     [ebitda_hist[-1]/revenue[-1]] + ebitda_margin_proj, FMT_PCT2, True),
    ("D&A as % of Revenue",
     [da_vals[-1]/revenue[-1]] + da_pct_proj, FMT_PCT2, True),
    ("CapEx as % of Revenue",
     [abs(capex_cf[-1])/revenue[-1]] + capex_pct_proj, FMT_PCT2, True),
    ("Δ NWC as % of Revenue",
     [0] + nwc_chg_pct, FMT_PCT2, True),
    ("Effective Tax Rate %",
     [tax_rate_wacc] + tax_proj, FMT_PCT2, True),
]

for ii, (lbl, vals, fmt, is_inp) in enumerate(assump_rows):
    alt = ii % 2 == 0
    label_cell(ws_dcf, r, 1, lbl, alt_row=alt)
    for ci, v in enumerate(vals):
        dc = data_cell(ws_dcf, r, ci+2, safe(v), fmt=fmt, alt_row=alt,
                       is_input=(is_inp and ci > 0))
    r += 1

r += 1

# ── PROJECTED P&L + FCFF ──────────────────────────────────────
section_header(ws_dcf, r, 1, 10,
    "SECTION 3 — PROJECTED INCOME STATEMENT & FREE CASH FLOW TO FIRM ($mm)")
r += 1

col_header(ws_dcf, r, 1, "Line Item ($mm)", bg=C_GREY_HDR, align=AL_LEFT)
for i, yr in enumerate(all_years_dcf):
    col_header(ws_dcf, r, i+2, yr)
r += 1

proj_rows_full = [
    ("Revenue",           [revenue[-1]]     + proj_rev,    FMT_MM,  True),
    ("  YoY Growth",      [safe(rev_growth[-1])/100 if safe(rev_growth[-1]) else None]
                          + rev_growth_proj,               FMT_PCT1,False),
    ("EBITDA",            [ebitda_hist[-1]] + proj_ebitda, FMT_MM,  True),
    ("  EBITDA Margin",   [ebitda_hist[-1]/revenue[-1]]
                          + ebitda_margin_proj,            FMT_PCT1,False),
    ("  Less: D&A",       [da_vals[-1]]
                          + proj_da,                       FMT_MM,  False),
    ("EBIT",              [ebit[-1]]        + proj_ebit,   FMT_MM,  True),
    ("  Less: Taxes (NOPAT)", [nopat_hist[-1]] + proj_nopat, FMT_MM,False),
    ("NOPAT",             [nopat_hist[-1]]  + proj_nopat,  FMT_MM,  True),
    ("  Add: D&A",        [da_vals[-1]]     + proj_da,     FMT_MM,  False),
    ("  Less: CapEx",     [abs(capex_cf[-1])]
                          + [proj_rev[i]*capex_pct_proj[i] for i in range(N_PROJ)], FMT_MM, False),
    ("  Less: Δ NWC",     [0]
                          + proj_nwc,                      FMT_MM,  False),
    ("FREE CASH FLOW TO FIRM (FCFF)", [safe(fcff_hist[-1])] + proj_fcff, FMT_MM, True),
    ("  FCF Margin %",    [fcff_hist[-1]/revenue[-1] if safe(fcff_hist[-1]) and revenue[-1]!=0 else None]
                          + [proj_fcff[i]/proj_rev[i] for i in range(N_PROJ)], FMT_PCT1, False),
    ("  Discount Factor (WACC={:.2f}%)".format(wacc*100),
     [1.0] + [1/(1+wacc)**(i+1) for i in range(N_PROJ)],  '0.0000', False),
    ("  PV of FCFF",
     [None] + pv_fcff,                                     FMT_MM,  False),
]

bold_rows_dcf = {"Revenue","EBITDA","EBIT","NOPAT",
                 "FREE CASH FLOW TO FIRM (FCFF)"}

for ii, (lbl, vals, fmt, is_tot) in enumerate(proj_rows_full):
    alt = ii % 2 == 0
    ws_dcf.row_dimensions[r].height = 15
    is_section = lbl in bold_rows_dcf
    if is_section:
        lc = ws_dcf.cell(r, 1, lbl)
        lc.font  = Font("Calibri",9,True,color=C_WHITE)
        lc.fill  = fill(C_DARK_BLUE)
        lc.alignment = AL_LEFT; lc.border = thin_border()
        for ci, v in enumerate(vals):
            dc = ws_dcf.cell(r, ci+2, safe(v))
            dc.font  = Font("Calibri",9,True,color=C_WHITE)
            dc.fill  = fill(C_DARK_BLUE)
            dc.number_format = fmt; dc.alignment = AL_RT; dc.border = thin_border()
    else:
        label_cell(ws_dcf, r, 1, lbl, indent=2, alt_row=alt)
        for ci, v in enumerate(vals):
            data_cell(ws_dcf, r, ci+2, safe(v), fmt=fmt, alt_row=alt)
    r += 1

r += 1

# ── EV BRIDGE ─────────────────────────────────────────────────
section_header(ws_dcf, r, 1, 10,
    "SECTION 4 — ENTERPRISE VALUE → EQUITY VALUE BRIDGE ($mm)")
r += 1

bridge_rows = [
    ("Sum of PV of FCFFs (FY2026E–FY2030E)", sum_pv_fcff,   FMT_MM,  False, False),
    ("Terminal Value (Year 5 FCFF × (1+g)/(WACC-g))", tv,   FMT_MM,  False, False),
    ("PV of Terminal Value",                  pv_tv,         FMT_MM,  False, False),
    ("  TV as % of Enterprise Value",         pv_tv/ev_dcf,  FMT_PCT1,False, False),
    ("ENTERPRISE VALUE (DCF)",                ev_dcf,        FMT_MM,  True,  False),
    ("  (−) Total Debt",                     -total_debt[-1],FMT_MM,  False, True),
    ("  (+) Cash & Short-Term Securities",
     cash_eq[-1] + (mkt_sec_st[-1] if safe(mkt_sec_st[-1]) else 0),
     FMT_MM, False, False),
    ("EQUITY VALUE",                          equity_val,    FMT_MM,  True,  False),
    ("  Diluted Shares Outstanding (mm)",     shares_mm,     '#,##0.0',False,False),
    ("INTRINSIC VALUE PER SHARE (DCF)",       intrinsic_ps,  FMT_PS,  True,  False),
    ("Current Market Price",                  current_price, FMT_PS,  False, False),
    ("Implied Upside / (Downside)",
     intrinsic_ps/current_price-1,            FMT_PCT1, True, False),
]

col_header(ws_dcf, r, 1, "Component", bg=C_GREY_HDR, align=AL_LEFT)
col_header(ws_dcf, r, 2, "Value ($mm / $/sh)", bg=C_GREY_HDR)
for c in range(3,11):
    ws_dcf.cell(r,c).fill=fill(C_GREY_HDR); ws_dcf.cell(r,c).border=thin_border()
r += 1

for ii, (lbl, val, fmt, is_total, is_red) in enumerate(bridge_rows):
    alt = ii % 2 == 0
    bg  = C_LIGHT_BLU if alt else C_WHITE
    is_up = "Upside" in lbl
    clr_  = (C_GREEN if val > 0 else C_RED) if is_up else \
            (C_NAVY if is_total else (C_RED if is_red else C_BLACK))
    bold_ = is_total or is_up

    lc = ws_dcf.cell(r, 1, lbl)
    lc.font  = Font("Calibri",9,bold_,color=clr_)
    lc.fill  = fill(bg); lc.alignment=AL_LEFT; lc.border=thin_border()
    ws_dcf.merge_cells(start_row=r,start_column=1,end_row=r,end_column=1)

    vc = ws_dcf.cell(r, 2, safe(val))
    vc.font  = Font("Calibri",9,bold_,color=clr_)
    vc.fill  = fill(bg); vc.number_format=fmt
    vc.alignment=AL_RT; vc.border=thin_border()
    ws_dcf.merge_cells(start_row=r,start_column=2,end_row=r,end_column=10)
    if is_total:
        vc.border=thick_border_bottom()
    r += 1

# ── Comparable Companies ──────────────────────────────────────
r += 1
section_header(ws_dcf, r, 1, 10,
    "SECTION 5 — COMPARABLE COMPANY ANALYSIS (BIG TECH PEERS)")
r += 1

col_headers_comp = ["Company","EV/EBITDA (NTM)","P/E (NTM)",
                    "Implied EV (EV/EBITDA)","Implied Price/Sh (P/E)","vs. Market"]
for ci, h in enumerate(col_headers_comp):
    col_header(ws_dcf, r, ci+1, h)
r += 1

peers = list(peer_ev_ebitda.keys())
for ii, peer in enumerate(peers):
    alt = ii % 2 == 0
    ev_m  = peer_ev_ebitda[peer] * ntm_ebitda
    pe_ps = peer_pe[peer] * ntm_eps
    vs_   = pe_ps / current_price - 1

    row_data = [
        peer,
        peer_ev_ebitda[peer],
        peer_pe[peer],
        ev_m,
        pe_ps,
        vs_,
    ]
    fmts = [None, FMT_MULT, FMT_MULT, FMT_MM, FMT_PS, FMT_PCT1]
    for ci, (v, f) in enumerate(zip(row_data, fmts)):
        dc = ws_dcf.cell(r, ci+1, v)
        dc.fill = fill(C_LIGHT_BLU if alt else C_WHITE)
        dc.border = thin_border()
        if f:
            dc.number_format = f
            dc.alignment = AL_RT
            is_up_cell = (ci == 5)
            dc.font = Font("Calibri", 9,
                           bold=(ci==0 or ci==5),
                           color=(C_GREEN if v>0 else C_RED) if is_up_cell
                                 else (C_NAVY if ci==0 else C_BLACK))
        else:
            dc.font = Font("Calibri",9,True,color=C_NAVY)
            dc.alignment = AL_LEFT
    r += 1

# Median row
med_ev_eb  = np.median(list(peer_ev_ebitda.values()))
med_pe     = np.median(list(peer_pe.values()))
med_ev     = med_ev_eb * ntm_ebitda
med_ps     = med_pe * ntm_eps
med_vs     = med_ps / current_price - 1

median_vals = ["PEER MEDIAN", med_ev_eb, med_pe, med_ev, med_ps, med_vs]
median_fmts = [None, FMT_MULT, FMT_MULT, FMT_MM, FMT_PS, FMT_PCT1]
for ci, (v, f) in enumerate(zip(median_vals, median_fmts)):
    dc = ws_dcf.cell(r, ci+1, v)
    dc.font  = Font("Calibri",9,True,color=C_WHITE)
    dc.fill  = fill(C_NAVY); dc.border=thin_border()
    if f:
        dc.number_format = f; dc.alignment=AL_RT
    else:
        dc.alignment = AL_LEFT
r += 2

print("✅ Sheet 6 (DCF Valuation) — done")

# ══════════════════════════════════════════════════════════════
# SHEET 7 — SENSITIVITY ANALYSIS
# ══════════════════════════════════════════════════════════════

ws_sen = wb.create_sheet("Sensitivity Analysis")
ws_sen.sheet_view.showGridLines = False
freeze_and_zoom(ws_sen, "B4", 85)
set_col_widths(ws_sen, {1:22, 2:14, 3:14, 4:14, 5:14,
                         6:14, 7:14, 8:22, 9:14, 10:14,
                         11:14, 12:14, 13:14})

# Title
ws_sen.row_dimensions[1].height = 28
ws_sen.merge_cells("A1:M1")
c = ws_sen.cell(1,1,
    "APPLE INC. — SENSITIVITY ANALYSIS | Intrinsic Value Per Share ($)")
c.font=Font("Calibri",12,True,color=C_WHITE)
c.fill=fill(C_NAVY); c.alignment=AL_CTR
ws_sen.row_dimensions[2].height = 5
for col in range(1,14): ws_sen.cell(2,col).fill=fill(C_ACCENT)

def write_sensitivity_table(ws, start_row, start_col,
                             row_vals, col_vals,
                             data_dict, row_label, col_label,
                             row_fmt, col_fmt, title):
    r = start_row
    # Table title
    end_c = start_col + len(col_vals)
    ws.merge_cells(start_row=r, start_column=start_col,
                   end_row=r,   end_column=end_c)
    tc = ws.cell(r, start_col, title)
    tc.font  = Font("Calibri",10,True,color=C_WHITE)
    tc.fill  = fill(C_NAVY); tc.alignment=AL_CTR; tc.border=outer_border()
    r += 1

    # Axis labels
    lbl_c = ws.cell(r, start_col,
                    f"↓ {row_label}   |   {col_label} →")
    lbl_c.font  = Font("Calibri",8,True,color=C_NAVY,italic=True)
    lbl_c.fill  = fill(C_GREY_HDR); lbl_c.alignment=AL_CTR
    lbl_c.border= thin_border()
    for ci, cv in enumerate(col_vals):
        ch = ws.cell(r, start_col+1+ci, cv)
        ch.number_format = col_fmt
        ch.font  = Font("Calibri",9,True,color=C_WHITE)
        ch.fill  = fill(C_DARK_BLUE)
        ch.alignment=AL_CTR; ch.border=thin_border()
    r += 1

    # Data rows
    min_v = min(data_dict.values())
    max_v = max(data_dict.values())

    for ri, rv in enumerate(row_vals):
        rh = ws.cell(r, start_col, rv)
        rh.number_format = row_fmt
        rh.font  = Font("Calibri",9,True,color=C_WHITE)
        rh.fill  = fill(C_DARK_BLUE)
        rh.alignment=AL_CTR; rh.border=thin_border()

        for ci, cv in enumerate(col_vals):
            val = data_dict.get((round(rv,4), round(cv,4)),
                  data_dict.get((round(rv,3), round(cv,3)), None))
            cell = ws.cell(r, start_col+1+ci, val)
            cell.number_format = FMT_PS
            cell.alignment = AL_CTR
            cell.border = thin_border()

            # Color gradient: red→white→green
            if val is not None and max_v != min_v:
                ratio = (val - min_v) / (max_v - min_v)
                if ratio < 0.5:
                    # red → white
                    t = ratio * 2
                    red   = 255
                    green = int(255 * t)
                    blue  = int(255 * t)
                else:
                    # white → green
                    t = (ratio - 0.5) * 2
                    red   = int(255 * (1 - t))
                    green = 255
                    blue  = int(255 * (1 - t))
                hex_c = f"{red:02X}{green:02X}{blue:02X}"
                cell.fill = fill(hex_c)
                # Text color for readability
                lum = 0.299*red + 0.587*green + 0.114*blue
                txt = C_WHITE if lum < 130 else C_BLACK
            else:
                txt = C_BLACK

            # Bold + highlight if it's the base case
            is_base = (abs(rv - list(data_dict.keys())[len(row_vals)//2][0]) < 1e-6 and
                       abs(cv - list(data_dict.keys())[len(col_vals)//2][1]) < 1e-6)
            cell.font = Font("Calibri", 9,
                             bold=is_base,
                             color=txt)
            if is_base:
                cell.border = outer_border()
        r += 1
    return r + 1

# TABLE 1 — WACC vs Terminal Growth Rate
r = 3
section_header(ws_sen, r, 1, 13,
    "TABLE 1 — WACC × Terminal Growth Rate  (Base: WACC={:.2f}%, TGR={:.1f}%)".format(
        wacc*100, tgr*100))
r += 1
r = write_sensitivity_table(
    ws_sen, r, 1,
    row_vals = wacc_range,
    col_vals = tgr_range,
    data_dict= sens_wacc_tgr,
    row_label= "WACC",
    col_label= "Terminal Growth Rate",
    row_fmt  = FMT_PCT2,
    col_fmt  = FMT_PCT2,
    title    = "Intrinsic Value / Share — WACC vs. TGR"
)

# TABLE 2 — Revenue Growth vs EBITDA Margin
r += 1
section_header(ws_sen, r, 1, 13,
    "TABLE 2 — Revenue Growth × EBITDA Margin  (Base: Growth=8.0%, Margin=35.8%)")
r += 1
r = write_sensitivity_table(
    ws_sen, r, 1,
    row_vals = rev_g_range,
    col_vals = marg_range,
    data_dict= sens_rev_marg,
    row_label= "Revenue CAGR",
    col_label= "EBITDA Margin",
    row_fmt  = FMT_PCT1,
    col_fmt  = FMT_PCT1,
    title    = "Intrinsic Value / Share — Revenue Growth vs. EBITDA Margin"
)

# Bull / Base / Bear scenarios
r += 1
section_header(ws_sen, r, 1, 13,
    "TABLE 3 — BULL / BASE / BEAR SCENARIO COMPARISON")
r += 1

scenarios = {
    "BEAR CASE":  {"rev_g": 0.04, "ebitda_m": 0.32, "wacc_s": wacc+0.01, "tgr_s": 0.02,
                   "color": C_RED},
    "BASE CASE":  {"rev_g": 0.08, "ebitda_m": 0.358,"wacc_s": wacc,      "tgr_s": 0.03,
                   "color": C_DARK_BLUE},
    "BULL CASE":  {"rev_g": 0.12, "ebitda_m": 0.39, "wacc_s": wacc-0.01, "tgr_s": 0.035,
                   "color": C_GREEN},
}

scen_headers = ["Scenario","Rev. CAGR","EBITDA Margin",
                "WACC","TGR","NTM Rev ($B)","NTM EBITDA ($B)",
                "EV ($B)","Net Debt ($B)","Equity Val ($B)",
                "Shares (B)","Price/Share","vs. Market"]
for ci, h in enumerate(scen_headers):
    col_header(ws_sen, r, ci+1, h)
r += 1

for sc_name, sc in scenarios.items():
    sc_fcff = []
    base_r  = revenue[-1]
    for i in range(N_PROJ):
        rv  = base_r * (1+sc["rev_g"])**(i+1)
        eb  = rv * sc["ebitda_m"]
        da_ = rv * da_pct_proj[i]
        np_ = (eb - da_) * (1 - tax_proj[i])
        cp_ = rv * capex_pct_proj[i]
        nw_ = rv * nwc_chg_pct[i]
        sc_fcff.append(np_ + da_ - cp_ - nw_)

    sc_tv = sc_fcff[-1]*(1+sc["tgr_s"])/(sc["wacc_s"]-sc["tgr_s"])
    sc_pv = sum([sc_fcff[i]/(1+sc["wacc_s"])**(i+1) for i in range(N_PROJ)])
    sc_ev = sc_pv + sc_tv/(1+sc["wacc_s"])**N_PROJ
    sc_eq = sc_ev - net_debt_cur
    sc_ps = sc_eq / shares_mm
    sc_vs = sc_ps / current_price - 1
    sc_ntm_rev  = revenue[-1] * (1+sc["rev_g"])
    sc_ntm_ebit = sc_ntm_rev  * sc["ebitda_m"]

    row_vals = [
        sc_name,
        sc["rev_g"],
        sc["ebitda_m"],
        sc["wacc_s"],
        sc["tgr_s"],
        sc_ntm_rev/1000,
        sc_ntm_ebit/1000,
        sc_ev/1000,
        net_debt_cur/1000,
        sc_eq/1000,
        shares_mm/1000,
        sc_ps,
        sc_vs,
    ]
    row_fmts = [None, FMT_PCT1, FMT_PCT1, FMT_PCT2, FMT_PCT2,
                FMT_MM_D, FMT_MM_D, FMT_MM_D, FMT_MM_D, FMT_MM_D,
                '0.00"B"', FMT_PS, FMT_PCT1]
    color = sc["color"]

    for ci, (v, f) in enumerate(zip(row_vals, row_fmts)):
        dc = ws_sen.cell(r, ci+1, v)
        dc.fill  = fill(C_LIGHT_BLU if ci%2==0 else C_WHITE)
        dc.border= thin_border()
        if f:
            dc.number_format = f; dc.alignment=AL_RT
            dc.font = Font("Calibri",9,
                           bold=(ci==11 or ci==12 or ci==0),
                           color=color if (ci==0 or ci==11 or ci==12) else C_BLACK)
        else:
            dc.font = Font("Calibri",9,True,color=C_WHITE)
            dc.fill = fill(color); dc.alignment=AL_LEFT
    r += 1

print("✅ Sheet 7 (Sensitivity Analysis) — done")

# ══════════════════════════════════════════════════════════════
# SHEET 8 — RATIO DASHBOARD
# ══════════════════════════════════════════════════════════════

ws_rat = wb.create_sheet("Ratio Dashboard")
ws_rat.sheet_view.showGridLines = False
freeze_and_zoom(ws_rat, "C5", 85)
set_col_widths(ws_rat, {1:32, **{i:12 for i in range(2,13)}})

ws_rat.row_dimensions[1].height = 28
ws_rat.merge_cells("A1:L1")
c = ws_rat.cell(1,1,
    "APPLE INC. — COMPREHENSIVE RATIO DASHBOARD | FY2016–FY2025 | Source: Bloomberg")
c.font=Font("Calibri",12,True,color=C_WHITE)
c.fill=fill(C_NAVY); c.alignment=AL_CTR
ws_rat.row_dimensions[2].height = 5
for col in range(1,13): ws_rat.cell(2,col).fill=fill(C_ACCENT)

r = 3
col_header(ws_rat, r, 1, "Ratio / Metric", bg=C_GREY_HDR, align=AL_LEFT)
for i, yr in enumerate(YEARS):
    col_header(ws_rat, r, i+2, yr)
r += 1

# Pull additional ratio data
ebitda_margin_hist = [ebitda_hist[i]/revenue[i] if safe(revenue[i]) and revenue[i]!=0 else None for i in range(N_HIST)]
gross_margin_hist  = [gross_profit[i]/revenue[i] if safe(revenue[i]) and revenue[i]!=0 else None for i in range(N_HIST)]
net_margin_hist    = [net_income[i]/revenue[i]   if safe(revenue[i]) and revenue[i]!=0 else None for i in range(N_HIST)]
ebit_margin_hist   = [ebit[i]/revenue[i]         if safe(revenue[i]) and revenue[i]!=0 else None for i in range(N_HIST)]
fcf_margin_hist    = [fcf_equity[i]/revenue[i]   if safe(revenue[i]) and revenue[i]!=0 else None for i in range(N_HIST)]

# DuPont deeper
asset_turnover_h   = [revenue[i]/total_assets[i] if safe(total_assets[i]) and total_assets[i]!=0 else None for i in range(N_HIST)]
equity_mult_h      = [total_assets[i]/tot_equity[i] if safe(tot_equity[i]) and tot_equity[i]!=0 else None for i in range(N_HIST)]
roe_calc_h         = [net_income[i]/tot_equity[i]*100 if safe(tot_equity[i]) and tot_equity[i]!=0 else None for i in range(N_HIST)]
roa_calc_h         = [net_income[i]/total_assets[i]*100 if safe(total_assets[i]) and total_assets[i]!=0 else None for i in range(N_HIST)]

# Leverage ratios
debt_equity_h      = [total_debt[i]/tot_equity[i] if safe(tot_equity[i]) and tot_equity[i]!=0 else None for i in range(N_HIST)]
debt_assets_h      = [total_debt[i]/total_assets[i] if safe(total_assets[i]) and total_assets[i]!=0 else None for i in range(N_HIST)]
net_debt_ebitda_h  = [(total_debt[i]-cash_eq[i])/ebitda_hist[i] if safe(ebitda_hist[i]) and ebitda_hist[i]!=0 else None for i in range(N_HIST)]
int_coverage_h     = [ebit[i]/int_exp[i] if safe(int_exp[i]) and int_exp[i]!=0 else None for i in range(N_HIST)]

# Dividend and capital return
payout_ratio_h     = [div_per_sh[i]/eps_diluted[i] if safe(eps_diluted[i]) and eps_diluted[i]!=0 else None for i in range(N_HIST)]
total_return_h     = [(abs(div_paid[i])+abs(buybacks[i]))/cfo[i] if safe(cfo[i]) and cfo[i]!=0 else None for i in range(N_HIST)]

ratio_sections = [
    # (section title, [(label, values, fmt)])
    ("PROFITABILITY", [
        ("Gross Margin %",          gross_margin_hist,  FMT_PCT1),
        ("EBITDA Margin %",         ebitda_margin_hist, FMT_PCT1),
        ("EBIT Margin %",           ebit_margin_hist,   FMT_PCT1),
        ("Net Profit Margin %",     net_margin_hist,    FMT_PCT1),
        ("FCF Margin %",            fcf_margin_hist,    FMT_PCT1),
        ("Return on Equity (ROE %)",roe_calc_h,         FMT_MM_D),
        ("Return on Assets (ROA %)",roa_calc_h,         FMT_MM_D),
        ("Return on Inv. Capital (ROIC %)", roic_calc,  FMT_MM_D),
    ]),
    ("GROWTH (Year-on-Year %)", [
        ("Revenue Growth %",        [v/100 if safe(v) else None for v in rev_growth],  FMT_PCT1),
        ("EBITDA Growth %",         [v/100 if safe(v) else None for v in ebitda_grw],  FMT_PCT1),
        ("EBIT Growth %",           [v/100 if safe(v) else None for v in ebit_grw],    FMT_PCT1),
        ("Net Income Growth %",     [v/100 if safe(v) else None for v in ni_grw],      FMT_PCT1),
        ("EPS (Diluted) Growth %",  [v/100 if safe(v) else None for v in eps_grw],     FMT_PCT1),
        ("FCF Growth %",
         [None] + [(fcf_equity[i]-fcf_equity[i-1])/abs(fcf_equity[i-1])
                    if safe(fcf_equity[i]) and safe(fcf_equity[i-1]) and fcf_equity[i-1]!=0
                    else None for i in range(1, N_HIST)], FMT_PCT1),
    ]),
    ("CREDIT & LEVERAGE", [
        ("Total Debt / EBITDA",     net_debt_ebitda_h,  FMT_MULT),
        ("Net Debt / EBITDA",       [v for v in net_debt_ebitda_h], FMT_MULT),
        ("Total Debt / Equity",     debt_equity_h,      FMT_MULT),
        ("Total Debt / Assets",     debt_assets_h,      FMT_PCT1),
        ("Interest Coverage (x)",   int_coverage_h,     FMT_MULT),
        ("Fixed Charge Coverage",   get_row(df_is, 75), FMT_MULT),
    ]),
    ("LIQUIDITY", [
        ("Current Ratio",           curr_ratio,         FMT_RATIO),
        ("Quick Ratio",             quick_ratio,        FMT_RATIO),
        ("Cash Ratio",              cash_ratio,         FMT_RATIO),
        ("Cash & Equiv. ($mm)",     cash_eq,            FMT_MM),
        ("CFO / Current Liabilities",
         [cfo[i]/curr_liab[i] if safe(curr_liab[i]) and curr_liab[i]!=0 else None
          for i in range(N_HIST)], FMT_RATIO),
    ]),
    ("WORKING CAPITAL EFFICIENCY", [
        ("Days Sales Outstanding",  dso,                '0.0"d"'),
        ("Inventory Days",          inv_days,           '0.0"d"'),
        ("Days Payable Outstanding",dpo,                '0.0"d"'),
        ("Cash Conversion Cycle",   ccc,                '0.0"d"'),
        ("Accounts Recv. Turnover", get_row(df_wc, 5),  FMT_MULT),
        ("Inventory Turnover",      get_row(df_wc, 7),  FMT_MULT),
    ]),
    ("DUPONT DECOMPOSITION", [
        ("Net Margin (Net Inc/Rev)",net_margin_hist,    FMT_PCT1),
        ("Asset Turnover (Rev/Assets)", asset_turnover_h, '0.00"x"'),
        ("Equity Multiplier (Assets/Eq)", equity_mult_h, '0.00"x"'),
        ("ROE = NM × AT × EM",     roe_calc_h,         FMT_MM_D),
        ("Tax Burden %",           [v/100 if safe(v) else None for v in tax_burden], FMT_PCT1),
    ]),
    ("CAPITAL ALLOCATION & YIELD", [
        ("CapEx / Revenue %",      [abs(capex_cf[i])/revenue[i] if safe(revenue[i]) and revenue[i]!=0 else None for i in range(N_HIST)], FMT_PCT1),
        ("CapEx / D&A",            get_row(df_cap, 13), FMT_MULT),
        ("FCF Yield (hist. mkt cap)", [fcf_equity[i]/mktcap_hist[i] if safe(mktcap_hist[i]) and mktcap_hist[i]!=0 else None for i in range(N_HIST)], FMT_PCT1),
        ("Dividend Payout Ratio",  payout_ratio_h,     FMT_PCT1),
        ("Total Shareholder Return %", [v/100 if safe(v) else None for v in get_row(df_yld, 25)], FMT_PCT1),
        ("Buybacks + Divs ($mm)",  [abs(div_paid[i])+abs(buybacks[i]) if safe(div_paid[i]) and safe(buybacks[i]) else None for i in range(N_HIST)], FMT_MM),
        ("Buyback + Div / CFO",    total_return_h,     FMT_PCT1),
    ]),
]

for sec_name, rows in ratio_sections:
    # Section header
    section_header(ws_rat, r, 1, 12, sec_name)
    r += 1
    for ii, (lbl, vals, fmt) in enumerate(rows):
        alt = ii % 2 == 0
        ws_rat.row_dimensions[r].height = 15
        label_cell(ws_rat, r, 1, lbl, alt_row=alt)
        for ci, v in enumerate(vals):
            is_neg_red = any(kw in lbl for kw in ["Debt","Leverage","Payout"])
            data_cell(ws_rat, r, ci+2, safe(v), fmt=fmt, alt_row=alt,
                      is_neg_red=is_neg_red)
        r += 1
    r += 1

print("✅ Sheet 8 (Ratio Dashboard) — done")

# ══════════════════════════════════════════════════════════════
# SHEET 9 — ESG SCORECARD
# ══════════════════════════════════════════════════════════════

ws_esg = wb.create_sheet("ESG Scorecard")
ws_esg.sheet_view.showGridLines = False
freeze_and_zoom(ws_esg, "C5", 85)
set_col_widths(ws_esg, {1:36, **{i:12 for i in range(2,13)}})

ws_esg.row_dimensions[1].height = 28
ws_esg.merge_cells("A1:L1")
c = ws_esg.cell(1,1,
    "APPLE INC. — ESG SCORECARD & SUSTAINABILITY METRICS | FY2016–FY2025 | Source: Bloomberg")
c.font=Font("Calibri",12,True,color=C_WHITE)
c.fill=fill(C_NAVY); c.alignment=AL_CTR
ws_esg.row_dimensions[2].height = 5
for col in range(1,13): ws_esg.cell(2,col).fill=fill(C_ACCENT)

r = 3
col_header(ws_esg, r, 1, "ESG Metric", bg=C_GREY_HDR, align=AL_LEFT)
for i, yr in enumerate(YEARS):
    col_header(ws_esg, r, i+2, yr)
r += 1

# GHG intensity (Scope 1+2 / Revenue)
ghg_intensity = []
for i in range(N_HIST):
    s1 = ghg_scope1[i] if safe(ghg_scope1[i]) else 0
    s2 = ghg_scope2[i] if safe(ghg_scope2[i]) else 0
    rev_v = revenue[i]
    if safe(rev_v) and rev_v != 0:
        ghg_intensity.append((s1 + s2) / (rev_v / 1000))  # per $B revenue
    else:
        ghg_intensity.append(None)

renew_pct = [renew_energy[i]/energy_total[i]*100 if safe(energy_total[i]) and energy_total[i]!=0 else None for i in range(N_HIST)]
women_board_pct = [women_board[i]/board_size[i]*100 if safe(board_size[i]) and board_size[i]!=0 else None for i in range(N_HIST)]

esg_sections = [
    ("ESG COMPOSITE SCORES (Bloomberg BESG)", [
        ("Overall ESG Score (BESG, /10)",      esg_total,   '0.00'),
        ("Environmental Pillar Score (/10)",    esg_env,     '0.00'),
        ("Social Pillar Score (/10)",           esg_soc,     '0.00'),
        ("Governance Pillar Score (/10)",       esg_gov,     '0.00'),
    ]),
    ("ENVIRONMENTAL — CLIMATE & EMISSIONS", [
        ("GHG Scope 1 (metric tons CO₂e, 000s)",  ghg_scope1, '#,##0.0'),
        ("GHG Scope 2 Market-Based (t CO₂e,000s)",get_row(df_esg,25), '#,##0.0'),
        ("GHG Scope 3 (metric tons CO₂e, 000s)",  ghg_scope3, '#,##0.00'),
        ("GHG Intensity (Scope1+2 / $B Rev)",      ghg_intensity, '0.000'),
        ("Total Energy Consumption (GWh)",         energy_total,  '#,##0.0'),
        ("Renewable Energy Use (GWh)",             renew_energy,  '#,##0.0'),
        ("Renewable Energy %",                     renew_pct,     FMT_PCT1),
        ("Total Water Withdrawal (ML)",            water_use,     '#,##0.0'),
    ]),
    ("ENVIRONMENTAL — WASTE & MATERIALS", [
        ("Total Waste Generated (metric tons)",    get_row(df_esg,43), '#,##0.0'),
        ("Waste Recycled (metric tons)",           get_row(df_esg,44), '#,##0.0'),
        ("Waste Sent to Landfill (metric tons)",   get_row(df_esg,45), '#,##0.0'),
        ("Hazardous Waste (metric tons)",          get_row(df_esg,42), '#,##0.00'),
    ]),
    ("SOCIAL — HUMAN CAPITAL", [
        ("Employees (headcount)",                  employees,         '#,##0'),
        ("% Women in Workforce",                   [v/100 if safe(v) else None for v in women_pct], FMT_PCT1),
        ("% Women in Mgmt",                        [v/100 if safe(v) else None for v in get_row(df_esg,75)], FMT_PCT1),
        ("Supplier Audits Conducted",              get_row(df_esg,94), '#,##0'),
    ]),
    ("GOVERNANCE", [
        ("Board Size",                             board_size,        '#,##0'),
        ("% Women on Board",                       women_board_pct,   FMT_PCT1),
        ("# Independent Directors",                get_row(df_esg,129),'#,##0'),
        ("Audit Committee Size",                   get_row(df_esg,100),'#,##0'),
        ("Audit Committee Meetings / Year",        get_row(df_esg,102),'#,##0'),
        ("Board Meeting Attendance %",             [v/100 if safe(v) else None for v in get_row(df_esg,111)], FMT_PCT1),
        ("Auditor Tenure (Years)",                 get_row(df_esg, 99),'#,##0'),
        ("Compensation Cmte Meetings / Year",      get_row(df_esg,118),'#,##0'),
    ]),
    ("ESG FINANCIAL MATERIALITY LINKAGE", [
        ("Revenue per Employee ($000s)",
         [revenue[i]/employees[i]*1000 if safe(employees[i]) and employees[i]!=0 else None
          for i in range(N_HIST)], '#,##0.0'),
        ("Net Income per Employee ($000s)",
         [net_income[i]/employees[i]*1000 if safe(employees[i]) and employees[i]!=0 else None
          for i in range(N_HIST)], '#,##0.0'),
        ("GHG Scope 3 / Revenue (t / $mm)",
         [ghg_scope3[i]/revenue[i] if safe(revenue[i]) and revenue[i]!=0 else None
          for i in range(N_HIST)], '0.000'),
        ("R&D Spend ($mm) — Innovation Proxy",     rd_exp,            FMT_MM),
        ("R&D % of Revenue",
         [rd_exp[i]/revenue[i] if safe(revenue[i]) and revenue[i]!=0 else None
          for i in range(N_HIST)], FMT_PCT1),
        ("SBC Expense ($mm) — Talent Retention",   get_row(df_cf,10), FMT_MM),
    ]),
]

C_ESG_E = "1A5C38"   # deep green — environmental
C_ESG_S = "0D4F8B"   # deep blue — social
C_ESG_G = "6B3A8C"   # deep purple — governance
C_ESG_F = "8B4513"   # brown — financial linkage

esg_sec_colors = [C_NAVY, C_ESG_E, C_ESG_E, C_ESG_S, C_ESG_G, C_DARK_BLUE]

for (sec_name, rows), sec_color in zip(esg_sections, esg_sec_colors):
    section_header(ws_esg, r, 1, 12, sec_name, bg=sec_color)
    r += 1
    for ii, (lbl, vals, fmt) in enumerate(rows):
        alt = ii % 2 == 0
        ws_esg.row_dimensions[r].height = 15
        label_cell(ws_esg, r, 1, lbl, alt_row=alt)
        for ci, v in enumerate(vals):
            data_cell(ws_esg, r, ci+2, safe(v), fmt=fmt, alt_row=alt)
        r += 1
    r += 1

# ESG scoring legend
r += 1
section_header(ws_esg, r, 1, 12,
    "BLOOMBERG ESG SCORE INTERPRETATION GUIDE", bg=C_GREY_HDR)
ws_esg.cell(r,1).font = Font("Calibri",9,True,color=C_NAVY)
r += 1
legend = [
    ("Score 7.0–10.0", "Leaders — Best-in-class ESG practices",   C_GREEN),
    ("Score 5.0–6.9",  "Above Average — Strong disclosure & practices", C_DARK_BLUE),
    ("Score 3.0–4.9",  "Average — Moderate ESG integration",       "FFA500"),
    ("Score 0.0–2.9",  "Laggards — Limited ESG disclosure",        C_RED),
]
for sc_lbl, sc_desc, sc_col in legend:
    ws_esg.cell(r,1, sc_lbl).font  = Font("Calibri",9,True, color=C_WHITE)
    ws_esg.cell(r,1).fill          = fill(sc_col)
    ws_esg.cell(r,1).border        = thin_border()
    ws_esg.merge_cells(start_row=r,start_column=1,end_row=r,end_column=3)
    ws_esg.cell(r,4, sc_desc).font = Font("Calibri",9)
    ws_esg.cell(r,4).border        = thin_border()
    ws_esg.merge_cells(start_row=r,start_column=4,end_row=r,end_column=12)
    r += 1

print("✅ Sheet 9 (ESG Scorecard) — done")

# ══════════════════════════════════════════════════════════════
# GLOBAL WORKBOOK SETTINGS & SHEET TAB COLORS
# ══════════════════════════════════════════════════════════════

tab_colors = {
    "Cover":               "1B3A6B",
    "Executive Summary":   "E8B84B",
    "Historical IS":       "2E5FA3",
    "Historical BS":       "2E5FA3",
    "Historical CF":       "2E5FA3",
    "DCF Valuation":       "217346",
    "Sensitivity Analysis":"C00000",
    "Ratio Dashboard":     "6B3A8C",
    "ESG Scorecard":       "1A5C38",
}
for sh_name, tab_col in tab_colors.items():
    if sh_name in wb.sheetnames:
        wb[sh_name].sheet_properties.tabColor = tab_col

# Set Cover as active sheet on open
wb.active = wb["Cover"]

# ══════════════════════════════════════════════════════════════
# SAVE WORKBOOK
# ══════════════════════════════════════════════════════════════

wb.save(OUTPUT_FILE)
print()
print("=" * 60)
print("  ✅ WORKBOOK SAVED SUCCESSFULLY")
print(f"  📁 File: {OUTPUT_FILE}")
print("=" * 60)
print()
print("  SHEETS INCLUDED:")
for i, sn in enumerate(wb.sheetnames, 1):
    print(f"    {i:02d}. {sn}")
print()
print("  MODEL SUMMARY:")
print(f"    Current Price      : ${current_price:.2f}")
print(f"    Intrinsic Value    : ${intrinsic_ps:.2f}")
print(f"    Upside/(Downside)  : {(intrinsic_ps/current_price-1)*100:+.1f}%")
print(f"    WACC               : {wacc*100:.2f}%")
print(f"    Terminal Gr. Rate  : {tgr*100:.1f}%")
print(f"    Enterprise Value   : ${ev_dcf/1e6:.2f}T")
print(f"    Equity Value       : ${equity_val/1e6:.2f}T")
print()

# Download trigger for Colab
try:
    from google.colab import files
    files.download(OUTPUT_FILE)
    print("  📥 Download started automatically.")
except Exception:
    print(f"  ℹ️  Run: files.download('{OUTPUT_FILE}') to download manually.")

✅ Sheet 6 (DCF Valuation) — done
✅ Sheet 7 (Sensitivity Analysis) — done
✅ Sheet 8 (Ratio Dashboard) — done
✅ Sheet 9 (ESG Scorecard) — done

  ✅ WORKBOOK SAVED SUCCESSFULLY
  📁 File: Apple_McKinsey_Valuation.xlsx

  SHEETS INCLUDED:
    01. Cover
    02. Executive Summary
    03. Historical IS
    04. Historical BS
    05. Historical CF
    06. DCF Valuation
    07. Sensitivity Analysis
    08. Ratio Dashboard
    09. ESG Scorecard

  MODEL SUMMARY:
    Current Price      : $254.23
    Intrinsic Value    : $136.25
    Upside/(Downside)  : -46.4%
    WACC               : 10.03%
    Terminal Gr. Rate  : 3.0%
    Enterprise Value   : $2.10T
    Equity Value       : $2.04T



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  📥 Download started automatically.


In [9]:
# ============================================================
# PATCH v2 — Fixed unmerge logic (no regex row parsing error)
# ============================================================

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import numpy as np

PATCHED_FILE = "Apple_McKinsey_Valuation_v2.xlsx"
wb = load_workbook("Apple_McKinsey_Valuation.xlsx")
ws = wb["DCF Valuation"]

# ── Style helpers ─────────────────────────────────────────────
C_NAVY      = "1B3A6B"; C_DARK_BLUE = "2E5FA3"; C_LIGHT_BLU = "D6E4F0"
C_ACCENT    = "E8B84B"; C_GREEN     = "217346"; C_RED       = "C00000"
C_GREY_HDR  = "F2F2F2"; C_WHITE     = "FFFFFF"; C_BLACK     = "000000"
C_BLUE_TXT  = "0000FF"

FMT_PCT2 = '0.00%;(0.00%);"-"'; FMT_PCT1 = '0.0%;(0.0%);"-"'
FMT_MM   = '#,##0;(#,##0);"-"'; FMT_PS   = '$#,##0.00'

def tfill(h): return PatternFill("solid", fgColor=h)
def tborder():
    s = Side(style='thin', color="BDD7EE")
    return Border(top=s, bottom=s, left=s, right=s)
def thick_bot():
    return Border(bottom=Side(style='medium', color=C_NAVY))
def outer_brd():
    s = Side(style='medium', color=C_NAVY)
    return Border(top=s, bottom=s, left=s, right=s)
def AL(h='left'): return Alignment(horizontal=h, vertical='center')

# ── WACC values ───────────────────────────────────────────────
rf_rate       = 0.0430;  erp = 0.0480;  beta = 1.24
ke            = rf_rate + beta * erp
kd_pretax     = 0.0310;  tax_rate_wacc = 0.156
kd_aftertax   = kd_pretax * (1 - tax_rate_wacc)
tgr           = 0.030
current_price = 254.23;  shares_mm = 15004.697
total_debt_bb = 112377.0; cash_plus_sec = 54697.0
market_cap_mm = current_price * shares_mm
total_cap     = market_cap_mm + total_debt_bb
wd            = total_debt_bb / total_cap
we            = market_cap_mm / total_cap
wacc          = we * ke + wd * kd_aftertax

proj_fcff     = [126664.84, 138533.46, 150143.57, 160233.96, 170169.77]
pv_fcff_list  = [proj_fcff[i]/(1+wacc)**(i+1) for i in range(5)]
sum_pv_fcff   = sum(pv_fcff_list)
tv            = proj_fcff[-1]*(1+tgr)/(wacc-tgr)
pv_tv         = tv/(1+wacc)**5
ev_dcf        = sum_pv_fcff + pv_tv
equity_val    = ev_dcf - total_debt_bb + cash_plus_sec
intrinsic_ps  = equity_val / shares_mm

print(f"WACC: {wacc*100:.4f}%  |  Ke: {ke*100:.3f}%  |  Kd(at): {kd_aftertax*100:.3f}%")
print(f"Intrinsic/Share: ${intrinsic_ps:.2f}  vs  Market: ${current_price:.2f}")

# ══════════════════════════════════════════════════════════════
# STEP A — Safely unmerge only rows 5-11
# ══════════════════════════════════════════════════════════════

# ✅ FIX: use .min_row / .max_row directly — no regex needed
ranges_to_remove = []
for mc in list(ws.merged_cells.ranges):
    if 5 <= mc.min_row <= 11 and mc.max_row <= 11:
        ranges_to_remove.append(str(mc))

print(f"Unmerging {len(ranges_to_remove)} ranges in rows 5-11...")
for rng in ranges_to_remove:
    try:
        ws.unmerge_cells(rng)
    except Exception as e:
        print(f"  Skipping {rng}: {e}")

# Clear cells rows 5-11, cols 1-10
for row_i in range(5, 12):
    for col_i in range(1, 11):
        c = ws.cell(row_i, col_i)
        c.value = None
        c.fill  = tfill(C_WHITE)
        c.border = tborder()

# ══════════════════════════════════════════════════════════════
# STEP B — Rewrite WACC section
# Layout: A:D = left label | E = left value | F = spacer | G:I = right label | J = right value
# ══════════════════════════════════════════════════════════════

left_rows = [
    ("Risk-Free Rate  (US 10Y Treasury)",         rf_rate,       FMT_PCT2, True),
    ("Equity Risk Premium  (Damodaran ERP)",       erp,           FMT_PCT2, True),
    ("Beta  (5Y Monthly vs S&P 500)",              beta,          '0.00',   True),
    ("Cost of Equity  (CAPM = Rf + β × ERP)",     ke,            FMT_PCT2, False),
    ("Pre-Tax Cost of Debt  (blended coupon)",     kd_pretax,     FMT_PCT2, True),
    ("Effective Tax Rate  (FY2025 Bloomberg)",     tax_rate_wacc, FMT_PCT2, False),
    ("After-Tax Cost of Debt  = Kd × (1 − t)",    kd_aftertax,   FMT_PCT2, False),
]
right_rows = [
    ("Market Capitalisation  ($mm)",               market_cap_mm, FMT_MM,   False),
    ("Total Debt  (S+LT)  ($mm)",                  total_debt_bb, FMT_MM,   False),
    ("Total Capital  ($mm)",                       total_cap,     FMT_MM,   False),
    ("Equity Weight  (We)",                        we,            FMT_PCT2, False),
    ("Debt Weight  (Wd)",                          wd,            FMT_PCT2, False),
    ("WACC  =  Ke × We  +  Kd(at) × Wd",         wacc,          FMT_PCT2, False),
    ("Terminal Growth Rate  (TGR)",                tgr,           FMT_PCT2, True),
]

for ii, row_i in enumerate(range(5, 12)):
    alt = (ii % 2 == 0)
    bg  = C_LIGHT_BLU if alt else C_WHITE

    lbl, val, fmt, is_inp = left_rows[ii]
    is_wacc_l = ("WACC" in lbl and "=" in lbl)
    row_bg    = C_LIGHT_BLU if is_wacc_l else bg

    # Left label  A:D
    lc = ws.cell(row_i, 1, lbl)
    lc.font      = Font("Calibri", 9, bold=is_wacc_l,
                         color=C_NAVY if is_wacc_l else C_BLACK)
    lc.fill      = tfill(row_bg); lc.alignment = AL('left'); lc.border = tborder()
    ws.merge_cells(start_row=row_i, start_column=1, end_row=row_i, end_column=4)

    # Left value  E
    vc = ws.cell(row_i, 5, val)
    vc.font          = Font("Calibri", 10, bold=is_wacc_l,
                             color=C_BLUE_TXT if is_inp else
                                   (C_NAVY if is_wacc_l else C_BLACK))
    vc.fill          = tfill(row_bg); vc.number_format = fmt
    vc.alignment     = AL('right')
    vc.border        = outer_brd() if is_wacc_l else tborder()

    # Spacer  F
    sp = ws.cell(row_i, 6); sp.fill = tfill(C_WHITE); sp.border = tborder()

    lbl2, val2, fmt2, is_inp2 = right_rows[ii]
    is_wacc_r = ("WACC" in lbl2 and "=" in lbl2)
    row_bg2   = C_LIGHT_BLU if is_wacc_r else bg

    # Right label  G:I
    lc2 = ws.cell(row_i, 7, lbl2)
    lc2.font      = Font("Calibri", 9, bold=is_wacc_r,
                          color=C_NAVY if is_wacc_r else C_BLACK)
    lc2.fill      = tfill(row_bg2); lc2.alignment = AL('left'); lc2.border = tborder()
    ws.merge_cells(start_row=row_i, start_column=7, end_row=row_i, end_column=9)

    # Right value  J
    vc2 = ws.cell(row_i, 10, val2)
    vc2.font          = Font("Calibri", 10, bold=is_wacc_r,
                              color=C_BLUE_TXT if is_inp2 else
                                    (C_NAVY if is_wacc_r else C_BLACK))
    vc2.fill          = tfill(row_bg2); vc2.number_format = fmt2
    vc2.alignment     = AL('right')
    vc2.border        = outer_brd() if is_wacc_r else tborder()

print("✅ STEP A+B — WACC section written")

# ══════════════════════════════════════════════════════════════
# STEP C — Fix EV Bridge rows 49, 51, 53
# ══════════════════════════════════════════════════════════════

bridge_fixes = {
    49: ("EQUITY VALUE",                 equity_val,                   FMT_MM,  C_NAVY),
    51: ("INTRINSIC VALUE PER SHARE",    intrinsic_ps,                 FMT_PS,  C_NAVY),
    53: ("Implied Upside / (Downside)",  intrinsic_ps/current_price-1, FMT_PCT1,
         C_GREEN if intrinsic_ps > current_price else C_RED),
}

for row_i, (lbl, val, fmt, clr) in bridge_fixes.items():
    try: ws.unmerge_cells(f"B{row_i}:J{row_i}")
    except: pass
    la = ws.cell(row_i, 1, lbl)
    la.font = Font("Calibri", 9, True, color=clr)
    la.border = tborder(); la.fill = tfill(C_LIGHT_BLU)
    vc = ws.cell(row_i, 2, val)
    vc.font          = Font("Calibri", 10, True, color=clr)
    vc.fill          = tfill(C_LIGHT_BLU); vc.number_format = fmt
    vc.alignment     = AL('right'); vc.border = thick_bot()
    ws.merge_cells(start_row=row_i, start_column=2, end_row=row_i, end_column=10)

print("✅ STEP C — EV Bridge fixed")

# ══════════════════════════════════════════════════════════════
# STEP D — Annotate PV of FCFF row (row 38)
# ══════════════════════════════════════════════════════════════

try: ws.unmerge_cells("A38")
except: pass

note = ws.cell(38, 1, "  PV of FCFF  (FY2026E–FY2030E, discounted at WACC)")
note.font = Font("Calibri", 9, italic=True, color=C_DARK_BLUE)
note.fill = tfill(C_WHITE); note.alignment = AL('left'); note.border = tborder()

dash = ws.cell(38, 2, "n/a (FY2025A)")
dash.font = Font("Calibri", 8, italic=True, color="888888")
dash.fill = tfill(C_WHITE); dash.alignment = AL('center'); dash.border = tborder()

print("✅ STEP D — PV row annotated")

# ══════════════════════════════════════════════════════════════
# STEP E — Analyst context box (rows 64 onwards)
# ══════════════════════════════════════════════════════════════

# Bull case DCF
wacc_bull = 0.085; tgr_bull = 0.035
pv_bull   = sum([proj_fcff[i]/(1+wacc_bull)**(i+1) for i in range(5)])
tv_bull   = proj_fcff[-1]*(1+tgr_bull)/(wacc_bull-tgr_bull)
ev_bull   = pv_bull + tv_bull/(1+wacc_bull)**5
ps_bull   = (ev_bull - total_debt_bb + cash_plus_sec) / shares_mm

# Peer implied price
ntm_ebitda = 161649.84
peer_ps    = (20.5 * ntm_ebitda - (total_debt_bb - cash_plus_sec)) / shares_mm

ctx = 64
# Clear rows 64-82 cleanly
for r_i in range(ctx, ctx+20):
    for c_i in range(1, 11):
        cell = ws.cell(r_i, c_i)
        cell.value = None; cell.fill = tfill(C_WHITE); cell.border = tborder()

# Section header
ws.merge_cells(start_row=ctx, start_column=1, end_row=ctx, end_column=10)
hdr = ws.cell(ctx, 1, "VALUATION CROSS-CHECK — MULTI-METHOD PRICE COMPARISON & ANALYST CONTEXT")
hdr.font = Font("Calibri",10,True,color=C_WHITE); hdr.fill = tfill(C_NAVY)
hdr.alignment = AL('center'); hdr.border = outer_brd()
ws.row_dimensions[ctx].height = 18
ctx += 1

# Column headers
for ci, (h, col_s, col_e) in enumerate([
    ("Valuation Method", 1, 3), ("Implied $/Share", 4, 5),
    ("vs. Market", 6, 6), ("Analyst Note", 7, 10)
]):
    ws.merge_cells(start_row=ctx, start_column=col_s, end_row=ctx, end_column=col_e)
    hc = ws.cell(ctx, col_s, h)
    hc.font = Font("Calibri",9,True,color=C_NAVY); hc.fill = tfill(C_GREY_HDR)
    hc.alignment = AL('center'); hc.border = tborder()
ws.row_dimensions[ctx].height = 16; ctx += 1

# Data rows
scenarios = [
    ("DCF Base Case  (WACC={:.2f}%, TGR=3.0%)".format(wacc*100),
     intrinsic_ps, intrinsic_ps/current_price-1,
     "Conservative 5-yr FCFF model. Reflects actual Bloomberg FY2025 financials.",
     C_RED if intrinsic_ps < current_price else C_GREEN),
    ("DCF Bull Case  (WACC=8.50%, TGR=3.5%)",
     ps_bull, ps_bull/current_price-1,
     "Lower WACC for Services durability + higher TGR for AI optionality.",
     C_GREEN if ps_bull > current_price else C_DARK_BLUE),
    ("Peer EV/EBITDA Median  (20.5× NTM EBITDA)",
     peer_ps, peer_ps/current_price-1,
     "NTM EBITDA = $161.6B. Peer set: MSFT, GOOGL, META, AMZN.",
     C_DARK_BLUE),
    ("Current Market Price  (Bloomberg 18-Mar-2026)",
     current_price, 0.0,
     "Market cap ~$3.82T. Reflects Services re-rating + ecosystem premium.",
     C_NAVY),
]

for ii, (meth, ps_v, vs_v, note_t, clr) in enumerate(scenarios):
    alt = ii % 2 == 0
    bg  = C_LIGHT_BLU if alt else C_WHITE
    ws.row_dimensions[ctx].height = 18

    # Method
    ws.merge_cells(start_row=ctx, start_column=1, end_row=ctx, end_column=3)
    mc = ws.cell(ctx, 1, meth)
    mc.font = Font("Calibri",9,bold=(ii==3),color=clr)
    mc.fill = tfill(bg); mc.alignment = AL('left'); mc.border = tborder()

    # Price
    ws.merge_cells(start_row=ctx, start_column=4, end_row=ctx, end_column=5)
    pc = ws.cell(ctx, 4, ps_v)
    pc.font = Font("Calibri",10,True,color=clr); pc.fill = tfill(bg)
    pc.number_format = FMT_PS; pc.alignment = AL('center'); pc.border = tborder()

    # Upside
    vc_c = ws.cell(ctx, 6, vs_v)
    vc_c.font = Font("Calibri",9,True,
                      color=C_GREEN if vs_v > 0 else (C_NAVY if vs_v==0 else C_RED))
    vc_c.fill = tfill(bg); vc_c.number_format = FMT_PCT1
    vc_c.alignment = AL('center'); vc_c.border = tborder()

    # Note
    ws.merge_cells(start_row=ctx, start_column=7, end_row=ctx, end_column=10)
    nc = ws.cell(ctx, 7, note_t)
    nc.font = Font("Calibri",8,italic=True,color="555555"); nc.fill = tfill(bg)
    nc.alignment = Alignment(horizontal='left',vertical='center',wrap_text=True)
    nc.border = tborder()
    ctx += 1

# Key insight paragraph
ctx += 1
ws.merge_cells(start_row=ctx, start_column=1, end_row=ctx, end_column=10)
ins = ws.cell(ctx, 1,
    "📌  KEY INSIGHT:  Apple's ~{:.0f}% market premium over DCF base case reflects: "
    "(1) Services segment re-rating toward SaaS multiples (gross margin >70%),  "
    "(2) Apple Intelligence / AI integration optionality not captured in 5-yr FCFF,  "
    "(3) Installed base lock-in across 2B+ active devices,  "
    "(4) Capital return discipline — $106B returned to shareholders in FY2025.  "
    "Bull-case DCF of ${:.2f} at WACC=8.5% is closer to market consensus.".format(
        (current_price/intrinsic_ps - 1)*100, ps_bull))
ins.font      = Font("Calibri",9,italic=True,color=C_NAVY)
ins.fill      = tfill("EBF5FB")
ins.alignment = Alignment(horizontal='left',vertical='center',wrap_text=True)
ins.border    = outer_brd()
ws.row_dimensions[ctx].height = 45
ctx += 2

# ── Color legend ──────────────────────────────────────────────
ws.merge_cells(start_row=ctx, start_column=1, end_row=ctx, end_column=10)
lg = ws.cell(ctx, 1, "COLOR CONVENTION  |  Blue = Input Assumption   Black = Calculated   Navy Bold = Subtotal   Green = Upside   Red = Downside")
lg.font = Font("Calibri",8,True,color=C_WHITE); lg.fill = tfill(C_NAVY)
lg.alignment = AL('center'); lg.border = tborder()
ws.row_dimensions[ctx].height = 14

print("✅ STEP E — Analyst context box written")

# ══════════════════════════════════════════════════════════════
# STEP F — Column widths & row heights
# ══════════════════════════════════════════════════════════════

col_w = {1:38, 2:13, 3:13, 4:13, 5:13, 6:3, 7:14, 8:14, 9:14, 10:14}
for c_i, w in col_w.items():
    ws.column_dimensions[get_column_letter(c_i)].width = w

for r_i in range(5, 12):
    ws.row_dimensions[r_i].height = 16

print("✅ STEP F — Column widths set")

# ══════════════════════════════════════════════════════════════
# SAVE
# ══════════════════════════════════════════════════════════════

wb.save(PATCHED_FILE)
print()
print("=" * 60)
print(f"  ✅  SAVED:  {PATCHED_FILE}")
print("=" * 60)
print(f"  WACC           : {wacc*100:.4f}%")
print(f"  Ke / Kd(at)    : {ke*100:.3f}% / {kd_aftertax*100:.3f}%")
print(f"  We / Wd        : {we*100:.1f}% / {wd*100:.1f}%")
print(f"  Intrinsic (Base): ${intrinsic_ps:.2f}  ({(intrinsic_ps/current_price-1)*100:+.1f}%)")
print(f"  Intrinsic (Bull): ${ps_bull:.2f}  ({(ps_bull/current_price-1)*100:+.1f}%)")
print(f"  Market Price    : ${current_price:.2f}")

try:
    from google.colab import files
    files.download(PATCHED_FILE)
    print(f"  📥 Download started.")
except:
    print(f"  ℹ️  Run files.download('{PATCHED_FILE}') to download.")

WACC: 10.0335%  |  Ke: 10.252%  |  Kd(at): 2.616%
Intrinsic/Share: $136.25  vs  Market: $254.23
Unmerging 28 ranges in rows 5-11...
✅ STEP A+B — WACC section written
✅ STEP C — EV Bridge fixed
✅ STEP D — PV row annotated
✅ STEP E — Analyst context box written
✅ STEP F — Column widths set

  ✅  SAVED:  Apple_McKinsey_Valuation_v2.xlsx
  WACC           : 10.0335%
  Ke / Kd(at)    : 10.252% / 2.616%
  We / Wd        : 97.1% / 2.9%
  Intrinsic (Base): $136.25  (-46.4%)
  Intrinsic (Bull): $190.99  (-24.9%)
  Market Price    : $254.23


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  📥 Download started.
